# Mapa de Cobertura Wi-Fi — Fases 5 e 6

**CC0048 — Redes Sem Fio** · UFCA · Prof. Laszlon Rodrigues da Costa

Este notebook implementa a **Fase 5 (Processamento)** e a **Fase 6 (Análise)** do projeto
de mapeamento de cobertura Wi-Fi, cobrindo múltiplas edificações.

## Como usar

1. Execute as células **em ordem, de cima para baixo**.
2. Para incluir uma nova edificação, preencha **apenas** a seção
   `>>> ADICIONAR BLOCO I AQUI <<<`. Nenhuma lógica de análise precisa ser tocada.
3. Blocos sem CSV disponível são ignorados com aviso; o notebook roda até o fim
   com os blocos que existirem.

## Princípios de implementação

| Princípio | Como é garantido |
|---|---|
| Nenhum nome de bloco no código de cálculo | Todas as funções recebem `bloco` como parâmetro |
| Descoberta dinâmica | Os blocos vêm de `BLOCOS.keys()`, nunca de lista fixa |
| Tolerância a ausência | `carregar_blocos()` pula CSV inexistente; seções checam pré-requisitos |
| Sem preenchimento silencioso | Coluna ausente → aviso + exclusão da análise, nunca valor default |
| Sem resultado sem confiança | Todo α vem com R², n, e guarda de suficiência amostral |

---
# Configuração

**Esta é a única célula que precisa ser editada para incluir dados novos.**

`BLOCOS` mapeia o identificador da edificação para tudo que a descreve: caminho do CSV,
planta, dimensões reais, posição dos APs, modelo do AP e data da campanha.

`PARAMS` concentra os limiares de decisão usados nas Fases 5 e 6.

In [ ]:
# =============================================================================
# CÉLULA DE CONFIGURAÇÃO ÚNICA
# =============================================================================
from pathlib import Path

BASE = Path.cwd()
DIR_DADOS = BASE / "dados"
DIR_SAIDA = BASE / "saida"
DIR_FIG   = DIR_SAIDA / "figuras"
DIR_QGIS  = DIR_SAIDA / "qgis"
for _d in (DIR_DADOS, DIR_SAIDA, DIR_FIG, DIR_QGIS):
    _d.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Schema canônico. O notebook inteiro trabalha sobre estes nomes de coluna.
# CSVs cujas colunas tenham outros nomes são normalizados via 'mapa_colunas'.
# -----------------------------------------------------------------------------
SCHEMA = [
    "bloco", "ponto", "andar", "local", "x", "y",
    "banda_ghz", "rssi_dbm", "snr_estimado_db",
    "canal", "canal_melhor", "qualidade_melhor_canal_pct",
    "redes_vizinhas_mesmo_canal", "distancia_ao_ap", "obstaculos",
    "throughput_tcp_mbps", "throughput_udp_mbps",
    "latencia_media_ms", "perda_pacotes_pct",
]

# -----------------------------------------------------------------------------
# BLOCOS — uma entrada por edificação.
# -----------------------------------------------------------------------------
BLOCOS = {

    "M": {
        # --- origem dos dados ---
        "csv": DIR_DADOS / "dados_bloco_M.csv",
        "data_campanha": "2026-08-07",          # 07/08/2026
        "modelo_ap": None,                       # não registrado em campo
        "n_leituras_campanha": 30,               # 30 leituras; 29 com conexão (ponto 22 = zona cega)

        # --- planta e geometria (necessários para os heatmaps da Fase 5.2) ---
        "planta": None,                          # ex.: DIR_DADOS / "planta_M.png"
        "dimensoes_m": None,                     # ex.: (40.0, 25.0)  -> (largura, altura) em metros
        "aps": {},                               # ex.: {"AP-M1": (12.0, 8.0)}

        # --- normalização do CSV bruto para o schema canônico ---
        "prefixo_ponto": "M",
        "mapa_colunas": {
            "ponto":  "ponto",
            "banda":  "banda_ghz",
            "rssi":   "rssi_dbm",
            "d":      "distancia_ao_ap",
            "obst":   "obstaculos",
            "canal":  "canal",
            "melhor": "canal_melhor",
            "pct":    "qualidade_melhor_canal_pct",
            "andar":  "andar",
        },

        # --- pontos excluídos do ajuste de path-loss, por banda, COM justificativa ---
        "exclusoes_ajuste": {
            2.4: {
                "pontos": [13, 14],
                "motivo": ("medidos a 1 m de um AP no Subsolo registrando -40 e -44 dBm, "
                           "contra -25 e -28 dBm em leituras equivalentes a 1 m no Térreo; "
                           "diferença de até 19 dB na mesma distância nominal não é explicável "
                           "por perda de percurso (provável associação a AP distinto — BSSID não registrado)"),
            },
        },
    },

}

# -----------------------------------------------------------------------------
# PARAMS — limiares e constantes de decisão.
# -----------------------------------------------------------------------------
PARAMS = {
    # --- Fase 6.1: modelo log-distância ---
    "d0_m": 1.0,                    # distância de referência FIXA (não derivada dos dados)
    "min_distancias_distintas": 3,  # guarda: abaixo disso, α não é estimável
    "min_pontos_ajuste": 3,         # guarda: abaixo disso, α não é estimável
    "alpha_teorico": (2.0, 4.0),    # faixa de referência para ambiente interno

    # --- Fase 6.1: teste de alavancagem (leave-one-out) ---
    "loo_delta_alpha": 0.3,         # remoção que altera α acima disso => ponto influente
    "loo_queda_r2": 0.2,            # remoção que derruba R² acima disso => ponto influente

    # --- SNR / SINR ---
    "piso_ruido_dbm": -95.0,        # piso de ruído ADOTADO (não medido), uniforme p/ todos os blocos
    "largura_canal_hz": 20e6,       # B = 20 MHz para a capacidade de Shannon

    # --- Fase 5.2: contornos de referência em RSSI ---
    "contornos_rssi": [-67.0, -70.0],
    "resolucao_malha": 200,         # células por eixo na malha de interpolação
    "idw_potencia": 2.0,            # expoente do IDW
    "metodo_interpolacao": "linear",# método padrão do griddata

    # --- Fase 6.2: classificação cruzada RSSI × throughput ---
    "rssi_bom_dbm": -67.0,          # acima disso, sinal considerado bom
    "throughput_baixo_mbps": 20.0,  # abaixo disso, vazão considerada baixa

    # --- validação ---
    "bandas_validas": {2.4, 5.0},
    "rssi_faixa_valida": (-100.0, -10.0),
}

print("Configuração carregada.")
print(f"Blocos declarados : {list(BLOCOS.keys())}")
print(f"Piso de ruído adotado : {PARAMS['piso_ruido_dbm']:.0f} dBm (uniforme, não medido)")
print(f"d0 fixo em {PARAMS['d0_m']:.0f} m")
print(f"Saídas em : {DIR_SAIDA}")

---
## >>> ADICIONAR BLOCO I AQUI <<<

Para incluir o Bloco I, **descomente o bloco abaixo e preencha os valores**. Nada mais
precisa ser alterado no notebook: a descoberta dos blocos é feita a partir de
`BLOCOS.keys()`, e toda a análise é parametrizada por bloco.

**Regras ao preencher**

- `csv` — se o arquivo ainda não existir, o notebook avisa e segue só com os blocos disponíveis.
- `mapa_colunas` — se o CSV já vier no schema canônico, use `{}` (dicionário vazio).
- `dimensoes_m` e `planta` — obrigatórios para os heatmaps da Fase 5.2. Sem eles, a seção é pulada.
- `modelo_ap` e `data_campanha` — alimentam o aviso de comparabilidade da Fase 6.4.
- `exclusoes_ajuste` — só preencha se houver ponto a excluir, e **sempre com o motivo escrito**.
- **Não invente coordenadas** `x`/`y` a partir da descrição textual do local. Se não foram
  medidas em planta, deixe as colunas vazias e o notebook bloqueia apenas os heatmaps.

In [ ]:
# =============================================================================
# >>> ADICIONAR BLOCO I AQUI <<<
# Descomente e preencha. Executar esta célula insere o bloco na configuração.
# =============================================================================

# BLOCOS["I"] = {
#     # --- origem dos dados ---
#     "csv": DIR_DADOS / "dados_bloco_I.csv",
#     "data_campanha": "AAAA-MM-DD",          # data da campanha de coleta
#     "modelo_ap": "PREENCHER",               # ex.: "Ubiquiti UniFi AC Pro"
#     "n_leituras_campanha": None,            # total de leituras realizadas
#
#     # --- planta e geometria (necessários para os heatmaps da Fase 5.2) ---
#     "planta": DIR_DADOS / "planta_I.png",   # imagem da planta baixa
#     "dimensoes_m": (0.0, 0.0),              # (largura, altura) reais em metros
#     "aps": {"AP-I1": (0.0, 0.0)},           # {nome: (x, y)} em metros, mesmo referencial
#
#     # --- normalização do CSV ---
#     "prefixo_ponto": "I",                   # gera IDs únicos: I-01, I-02, ...
#     "mapa_colunas": {},                     # vazio se o CSV já usa o schema canônico
#
#     # --- pontos excluídos do ajuste, por banda ---
#     "exclusoes_ajuste": {
#         # 2.4: {"pontos": [], "motivo": ""},
#     },
# }

print("Blocos atualmente configurados:", list(BLOCOS.keys()))

---
## Imports e utilitários de figura

A função `anotar_figura()` é o utilitário reutilizável exigido pelo enunciado: toda figura
recebe caixa de texto interna e rodapé pelo mesmo caminho de código, sem duplicação.
Ela escolhe automaticamente o canto **menos ocupado por dados** para posicionar a caixa.

In [ ]:
import textwrap
import unicodedata
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.interpolate import griddata

matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["savefig.dpi"] = 150
matplotlib.rcParams["font.size"] = 9
matplotlib.rcParams["axes.grid"] = True
matplotlib.rcParams["grid.alpha"] = 0.25

# Dependências opcionais -------------------------------------------------------
try:
    import rasterio
    from rasterio.transform import from_origin
    TEM_RASTERIO = True
except ImportError:
    TEM_RASTERIO = False

try:
    import pykrige  # noqa: F401
    TEM_PYKRIGE = True
except ImportError:
    TEM_PYKRIGE = False

print(f"rasterio disponível : {TEM_RASTERIO}  -> exportação QGIS usará "
      f"{'GeoTIFF' if TEM_RASTERIO else 'PNG + world file (.pgw)'}")
print(f"pykrige  disponível : {TEM_PYKRIGE}   -> krigagem "
      f"{'habilitada' if TEM_PYKRIGE else 'indisponível (usando griddata + IDW)'}")

# Registro global de artefatos gerados ----------------------------------------
FIGURAS_GERADAS = []
AVISOS = []
ANALISES_BLOQUEADAS = []

def avisar(msg, categoria="AVISO"):
    """Registra e imprime um aviso. Nunca lança exceção."""
    linha = f"[{categoria}] {msg}"
    AVISOS.append(linha)
    print(linha)

def bloquear(analise, bloco, motivo, colunas_faltantes=()):
    """Registra uma análise que não pôde ser executada por falta de dado."""
    ANALISES_BLOQUEADAS.append({
        "analise": analise, "bloco": bloco, "motivo": motivo,
        "colunas_necessarias": ", ".join(colunas_faltantes) if colunas_faltantes else "—",
    })
    print(f"[BLOQUEADO] {analise} | bloco {bloco}: {motivo}")

CARIMBO = datetime.now().strftime("%d/%m/%Y %H:%M")

def _pontos_do_eixo(ax):
    """Coordenadas de todos os dados desenhados, em fração dos eixos."""
    pts = []
    for col in ax.collections:
        try:
            off = np.asarray(col.get_offsets(), float)
            if off.size:
                pts.append(off)
        except Exception:
            pass
    for ln in ax.lines:
        xd = np.asarray(ln.get_xdata(), float); yd = np.asarray(ln.get_ydata(), float)
        if xd.size and yd.size and xd.size == yd.size:
            pts.append(np.column_stack([xd, yd]))
    for p in ax.patches:  # barras
        try:
            bb = p.get_bbox()
            pts.append(np.array([[bb.x0, bb.y0], [bb.x1, bb.y1]]))
        except Exception:
            pass
    if not pts:
        return np.empty((0, 2))
    P = np.vstack(pts)
    P = P[np.isfinite(P).all(axis=1)]
    y0, y1 = ax.get_ylim()
    if y1 == y0 or P.size == 0:
        return np.empty((0, 2))
    v = (P[:, 1] - y0) / (y1 - y0)
    return v

def _dim_eixos_pts(ax):
    """Largura e altura dos eixos em pontos tipográficos."""
    larg_pol, alt_pol = ax.figure.get_size_inches()
    pos = ax.get_position()
    return pos.width * larg_pol * 72.0, pos.height * alt_pol * 72.0

def _quebrar(texto, largura_chars):
    """Reflui cada linha lógica respeitando as quebras já escritas no texto."""
    saida = []
    for linha in texto.split("\n"):
        if not linha.strip():
            saida.append("")
            continue
        partes = textwrap.wrap(linha, width=max(40, int(largura_chars)),
                               subsequent_indent="   ", break_long_words=False,
                               break_on_hyphens=False)
        saida.extend(partes or [""])
    return "\n".join(saida)

def _anotar_externa(ax, texto, rodape, fontsize):
    """Caixa abaixo dos eixos, com a figura crescendo para acomodá-la.

    Usado em mapas: com aspecto igual e barra de cores, reservar faixa DENTRO dos
    eixos distorceria a geometria da planta.
    """
    fig = ax.figure
    larg_pol, alt_pol = fig.get_size_inches()
    largura_chars = (larg_pol * 72.0 * 0.94) / (0.55 * fontsize)
    texto = _quebrar(texto, largura_chars)
    n_linhas = texto.count("\n") + 1

    alt_caixa_pol = (n_linhas * 1.28 * fontsize + 14.0) / 72.0
    folga = 0.18
    nova_alt = alt_pol + alt_caixa_pol + folga

    # preserva o tamanho absoluto dos eixos, acrescentando espaço embaixo
    for eixo in fig.axes:
        pos = eixo.get_position()
        eixo.set_position([
            pos.x0,
            (pos.y0 * alt_pol + alt_caixa_pol + folga) / nova_alt,
            pos.width,
            pos.height * alt_pol / nova_alt,
        ])
    fig.set_size_inches(larg_pol, nova_alt)

    fig.text(0.012, (alt_caixa_pol * 0.5 + folga * 0.55) / nova_alt, texto,
             fontsize=fontsize, ha="left", va="center", linespacing=1.28,
             family="DejaVu Sans",
             bbox=dict(boxstyle="round,pad=0.42", facecolor="#FFFFFF",
                       edgecolor="#8A8A8A", alpha=0.88, linewidth=0.6))
    if rodape:
        fig.text(0.005, 0.002, rodape, fontsize=6.2, color="#4A4A4A",
                 ha="left", va="bottom")
    return ax

def anotar_figura(ax, texto, rodape=None, loc=None, fontsize=7.2, legenda_fontsize=7.2,
                  modo="faixa"):
    """Aplica a anotação padrão da figura, garantindo que a caixa não cubra dados.

    modo='faixa'   : reserva uma FAIXA dentro dos eixos, expandindo o limite de y de
                     modo que a área ocupada pela caixa fique vazia. A legenda, quando
                     existe, é realocada para a faixa oposta.
    modo='externa' : coloca a caixa abaixo dos eixos, aumentando a figura. Indicado para
                     mapas, onde o aspecto igual não admite expansão dos limites.

    texto  : explicação do gráfico + parâmetros do cálculo + fonte dos dados
    rodape : data de geração, CSV de origem, método de interpolação
    loc    : força 'lower'/'upper'; None = escolhe a metade menos ocupada
    """
    if modo == "externa":
        return _anotar_externa(ax, texto, rodape, fontsize)

    larg_pts, alt_pts = _dim_eixos_pts(ax)
    largura_chars = (larg_pts - 16.0) / (0.55 * fontsize)
    texto = _quebrar(texto, largura_chars)

    n_linhas = texto.count("\n") + 1
    alt_caixa = (n_linhas * 1.28 * fontsize + 10.0) / alt_pts      # fração dos eixos

    # metade menos ocupada por dados decide o lado da caixa
    if loc is None:
        v = _pontos_do_eixo(ax)
        if v.size:
            lado = "lower" if (v < 0.5).sum() <= (v >= 0.5).sum() else "upper"
        else:
            lado = "lower"
    else:
        lado = "lower" if "lower" in loc else "upper"

    # legenda: vai para a faixa oposta, e também recebe reserva
    leg = ax.get_legend()
    alt_leg = 0.0
    if leg is not None:
        # recupera do próprio objeto de legenda, para preservar handles customizados
        handles = list(getattr(leg, "legend_handles", None) or
                       getattr(leg, "legendHandles", []) or [])
        labels = [t.get_text() for t in leg.get_texts()]
        if not handles or len(handles) != len(labels):
            handles, labels = ax.get_legend_handles_labels()
        if handles:
            n_ent = len(handles)
            # a legenda ocupa só o lado direito: reserva parcial basta
            alt_leg = 0.55 * (n_ent * 1.45 * legenda_fontsize + 12.0) / alt_pts
            lado_leg = "upper" if lado == "lower" else "lower"
            ax.legend(handles, labels, loc=f"{lado_leg} right",
                      fontsize=legenda_fontsize, framealpha=0.92)

    # reserva de espaço: expande o eixo y para que a faixa da caixa fique vazia
    fb = (alt_caixa if lado == "lower" else alt_leg) + 0.025
    ft = (alt_caixa if lado == "upper" else alt_leg) + 0.025
    fb, ft = min(fb, 0.45), min(ft, 0.45)
    y0, y1 = ax.get_ylim()
    faixa = y1 - y0
    nova = faixa / max(1.0 - fb - ft, 0.25)
    ax.set_ylim(y0 - nova * fb, y1 + nova * ft)

    py = 0.008 if lado == "lower" else 0.992
    va = "bottom" if lado == "lower" else "top"
    ax.text(0.008, py, texto, transform=ax.transAxes, fontsize=fontsize,
            ha="left", va=va, linespacing=1.28, zorder=20, family="DejaVu Sans",
            bbox=dict(boxstyle="round,pad=0.42", facecolor="#FFFFFF",
                      edgecolor="#8A8A8A", alpha=0.88, linewidth=0.6))
    if rodape:
        ax.figure.text(0.005, 0.002, rodape, fontsize=6.2, color="#4A4A4A",
                       ha="left", va="bottom")
    return ax

def salvar_fig(fig, metrica, bloco, banda, andar="todos"):
    """Salva em PNG 150 dpi com o padrão fig_<metrica>_<bloco>_<banda>_<andar>.png"""
    def _limpa(s):
        s = str(s).replace("º", "o").replace("ª", "a")
        # remove acentos: nomes de arquivo ficam ASCII e portáveis
        s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
        return s.replace(" ", "-").replace("/", "-").replace(",", ".")
    banda_txt = f"{banda:g}GHz" if isinstance(banda, (int, float)) else _limpa(banda)
    nome = f"fig_{_limpa(metrica)}_{_limpa(bloco)}_{banda_txt}_{_limpa(andar)}.png"
    caminho = DIR_FIG / nome
    fig.savefig(caminho, dpi=150, bbox_inches="tight", facecolor="white")
    if nome not in FIGURAS_GERADAS:
        FIGURAS_GERADAS.append(nome)
    return nome

def rodape_padrao(cfg, extra=""):
    """Rodapé: data de geração + CSV de origem + método de interpolação."""
    csv = Path(cfg["csv"]).name if cfg.get("csv") else "—"
    partes = [f"Gerado em {CARIMBO}", f"fonte: {csv}"]
    if extra:
        partes.append(extra)
    return "  |  ".join(partes)

def fonte_dados(bloco, cfg, n):
    """Linha padronizada de proveniência para a caixa de texto."""
    data = cfg.get("data_campanha") or "data não informada"
    return f"Fonte: bloco {bloco}, {n} leituras, campanha de {data}"

print("Utilitários de figura prontos.")

---
# Fase 5.1 — Carga e validação

Carrega cada CSV declarado, normaliza para o schema canônico, valida tipos e faixas e
emite o **relatório de completude**: quais colunas estão preenchidas e, a partir disso,
quais análises ficam habilitadas ou bloqueadas.

Regras aplicadas:

- RSSI deve ser negativo e estar em faixa plausível;
- distância ao AP deve ser > 0;
- banda deve pertencer a {2.4, 5};
- identificador do ponto recebe prefixo do bloco (`M-01`, `I-01`), eliminando a colisão
  de numeração entre blocos;
- `sem_obstaculo` é derivado do texto de `obstaculos` — nunca assumido.

In [ ]:
TERMOS_SEM_OBSTACULO = {"nenhum", "nenhuma", "sem obstaculo", "sem obstáculo",
                        "livre", "-", "", "nan", "none"}

def _derivar_sem_obstaculo(serie):
    """True quando o texto de obstáculo indica ausência de obstrução."""
    return serie.fillna("").astype(str).str.strip().str.lower().isin(TERMOS_SEM_OBSTACULO)

def normalizar(df_bruto, bloco, cfg):
    """Renomeia colunas para o schema canônico e completa o que faltar com NaN."""
    df = df_bruto.rename(columns=cfg.get("mapa_colunas") or {}).copy()
    df["bloco"] = bloco
    for col in SCHEMA:
        if col not in df.columns:
            df[col] = np.nan
    # ID único por bloco
    pref = cfg.get("prefixo_ponto", bloco)
    df["ponto_num"] = pd.to_numeric(df["ponto"], errors="coerce")
    df["ponto_id"] = [f"{pref}-{int(p):02d}" if pd.notna(p) else f"{pref}-??"
                      for p in df["ponto_num"]]
    # tipos numéricos
    for col in ["x", "y", "banda_ghz", "rssi_dbm", "snr_estimado_db", "canal",
                "canal_melhor", "qualidade_melhor_canal_pct", "redes_vizinhas_mesmo_canal",
                "distancia_ao_ap", "throughput_tcp_mbps", "throughput_udp_mbps",
                "latencia_media_ms", "perda_pacotes_pct"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["andar"] = df["andar"].fillna("não informado").astype(str)
    df["sem_obstaculo"] = _derivar_sem_obstaculo(df["obstaculos"])
    # coerência com coluna 'livre' do CSV bruto, quando existir
    if "livre" in df_bruto.columns:
        livre_csv = df_bruto["livre"].astype(str).str.lower().isin({"true", "1", "sim"})
        divergentes = int((livre_csv.values != df["sem_obstaculo"].values).sum())
        if divergentes:
            avisar(f"bloco {bloco}: {divergentes} linha(s) com divergência entre a coluna "
                   f"'livre' do CSV e o texto de obstáculo; prevalece o texto.")
    return df

def validar(df, bloco):
    """Valida faixas e retorna o DataFrame sem as linhas inválidas."""
    problemas, mascara_ok = [], pd.Series(True, index=df.index)
    lo, hi = PARAMS["rssi_faixa_valida"]
    m = df["rssi_dbm"].notna() & ~df["rssi_dbm"].between(lo, hi)
    if m.any():
        problemas.append(f"{int(m.sum())} leitura(s) com RSSI fora de [{lo:.0f}, {hi:.0f}] dBm")
        mascara_ok &= ~m
    m = df["distancia_ao_ap"].notna() & (df["distancia_ao_ap"] <= 0)
    if m.any():
        problemas.append(f"{int(m.sum())} leitura(s) com distância <= 0")
        mascara_ok &= ~m
    m = df["banda_ghz"].notna() & ~df["banda_ghz"].isin(PARAMS["bandas_validas"])
    if m.any():
        problemas.append(f"{int(m.sum())} leitura(s) com banda fora de {PARAMS['bandas_validas']}")
        mascara_ok &= ~m
    m = df["rssi_dbm"].isna()
    if m.any():
        problemas.append(f"{int(m.sum())} leitura(s) sem RSSI (excluídas)")
        mascara_ok &= ~m
    for p in problemas:
        avisar(f"bloco {bloco}: {p}", "VALIDAÇÃO")
    if not problemas:
        print(f"[OK] bloco {bloco}: todas as {len(df)} linhas passaram na validação.")
    return df[mascara_ok].copy()

def carregar_blocos(config):
    """Carrega todos os blocos com CSV existente. Ausência não é erro."""
    dados = {}
    for bloco, cfg in config.items():
        caminho = Path(cfg["csv"]) if cfg.get("csv") else None
        if caminho is None or not caminho.exists():
            avisar(f"bloco {bloco}: CSV não encontrado em {caminho} — bloco ignorado "
                   f"nesta execução (seções comparativas serão puladas).", "AUSENTE")
            continue
        bruto = pd.read_csv(caminho)
        df = validar(normalizar(bruto, bloco, cfg), bloco)
        # SNR: usa o medido se houver; caso contrário deriva do piso adotado
        if df["snr_estimado_db"].notna().any():
            df["snr_origem"] = "informado no CSV"
        else:
            df["snr_estimado_db"] = df["rssi_dbm"] - PARAMS["piso_ruido_dbm"]
            df["snr_origem"] = f"derivado: RSSI - ({PARAMS['piso_ruido_dbm']:.0f} dBm)"
            avisar(f"bloco {bloco}: coluna snr_estimado_db ausente — SNR derivado do piso "
                   f"adotado de {PARAMS['piso_ruido_dbm']:.0f} dBm (valor NÃO medido).")
        dados[bloco] = df
        print(f"[CARGA] bloco {bloco}: {len(df)} leituras válidas de {caminho.name}")
    return dados

DADOS = carregar_blocos(BLOCOS)
BLOCOS_ATIVOS = list(DADOS.keys())   # descoberta dinâmica — nunca lista fixa
print(f"\nBlocos com dados nesta execução: {BLOCOS_ATIVOS}")
if not BLOCOS_ATIVOS:
    avisar("Nenhum bloco com dados. O notebook seguirá sem produzir análises.", "CRÍTICO")

In [ ]:
# --- Relatório de completude por bloco ---------------------------------------
# Cada análise declara de quais colunas depende. A completude decide o que roda.
DEPENDENCIAS = {
    "5.2 Heatmaps interpolados":        ["x", "y", "rssi_dbm"],
    "5.3 Camada de pontos QGIS":        ["x", "y"],
    "6.1 Ajuste de path-loss":          ["distancia_ao_ap", "rssi_dbm", "obstaculos"],
    "6.2 Cruzamento RSSI x throughput": ["rssi_dbm", "throughput_tcp_mbps"],
    "6.3 Shannon vs. medido":           ["snr_estimado_db", "throughput_tcp_mbps"],
    "6.4 Comparativo entre blocos":     ["distancia_ao_ap", "rssi_dbm"],
    "Análise de latência":              ["latencia_media_ms"],
    "Análise de perda de pacotes":      ["perda_pacotes_pct"],
    "Interferência co-canal medida":    ["redes_vizinhas_mesmo_canal"],
    "Descasamento de canal":            ["canal", "canal_melhor"],
}

def completude(df):
    """Fração de valores preenchidos por coluna do schema."""
    return {c: float(df[c].notna().mean()) if c in df.columns else 0.0 for c in SCHEMA}

def relatorio_completude(dados):
    linhas = []
    for bloco, df in dados.items():
        comp = completude(df)
        for col in SCHEMA:
            linhas.append({"bloco": bloco, "coluna": col,
                           "preenchido_pct": round(100 * comp[col], 1),
                           "situacao": "preenchida" if comp[col] > 0 else "VAZIA"})
    return pd.DataFrame(linhas)

def habilitacao(dados):
    """Para cada bloco x análise: habilitada ou bloqueada, e por qual coluna."""
    linhas = []
    for bloco, df in dados.items():
        comp = completude(df)
        for analise, cols in DEPENDENCIAS.items():
            faltando = [c for c in cols if comp.get(c, 0.0) == 0.0]
            linhas.append({
                "bloco": bloco, "analise": analise,
                "situacao": "HABILITADA" if not faltando else "BLOQUEADA",
                "colunas_faltantes": ", ".join(faltando) if faltando else "—",
            })
    return pd.DataFrame(linhas)

if BLOCOS_ATIVOS:
    TAB_COMPLETUDE = relatorio_completude(DADOS)
    TAB_HABILITACAO = habilitacao(DADOS)

    print("=" * 78)
    print("RELATÓRIO DE COMPLETUDE — colunas vazias por bloco")
    print("=" * 78)
    for bloco in BLOCOS_ATIVOS:
        sub = TAB_COMPLETUDE[TAB_COMPLETUDE.bloco == bloco]
        vazias = sub[sub.situacao == "VAZIA"].coluna.tolist()
        cheias = sub[sub.situacao == "preenchida"].coluna.tolist()
        print(f"\nBloco {bloco} — {len(DADOS[bloco])} leituras válidas")
        print(f"  preenchidas ({len(cheias)}): {', '.join(cheias)}")
        print(f"  VAZIAS      ({len(vazias)}): {', '.join(vazias) if vazias else '—'}")

    print("\n" + "=" * 78)
    print("HABILITAÇÃO DE ANÁLISES")
    print("=" * 78)
    display(TAB_HABILITACAO)
else:
    TAB_COMPLETUDE = pd.DataFrame()
    TAB_HABILITACAO = pd.DataFrame()

In [ ]:
# --- Interpretação da Fase 5.1 (texto para o relatório) ----------------------
print("=" * 78)
print("INTERPRETAÇÃO — Fase 5.1")
print("=" * 78)
for bloco in BLOCOS_ATIVOS:
    df, cfg = DADOS[bloco], BLOCOS[bloco]
    sub = TAB_HABILITACAO[TAB_HABILITACAO.bloco == bloco]
    hab = sub[sub.situacao == "HABILITADA"].analise.tolist()
    blq = sub[sub.situacao == "BLOQUEADA"]
    print(f"""
Bloco {bloco} — campanha de {cfg.get('data_campanha', 'data não informada')}
{'-' * 74}
A base contém {len(df)} leituras válidas distribuídas em {df.andar.nunique()} pavimento(s)
({', '.join(sorted(df.andar.unique()))}) e {df.banda_ghz.nunique()} faixa(s) de frequência
({', '.join(f'{b:g} GHz' for b in sorted(df.banda_ghz.dropna().unique()))}).
A faixa de RSSI observada vai de {df.rssi_dbm.min():.0f} a {df.rssi_dbm.max():.0f} dBm.
Das leituras, {int(df.sem_obstaculo.sum())} estão sem obstáculo declarado e
{int((~df.sem_obstaculo).sum())} apresentam obstrução — separação que sustenta a modelagem
de propagação da Fase 6.1.

Análises habilitadas ({len(hab)}): {', '.join(hab) if hab else 'nenhuma'}.
""")
    if len(blq):
        print(f"Análises bloqueadas ({len(blq)}) e a coluna que falta em cada uma:")
        for _, r in blq.iterrows():
            print(f"  - {r.analise:<34} falta: {r.colunas_faltantes}")
        print("""
Nenhum valor foi arbitrado para as colunas vazias: as leituras entram nas análises que
os dados sustentam e ficam de fora das demais, com o bloqueio declarado acima.""")

---
# Fase 5.2 — Interpolação e heatmaps

Malha fina via `numpy.meshgrid` sobre as **dimensões reais da planta**, interpolação com
`scipy.interpolate.griddata` e alternativa **IDW**.

**Por que o IDW existe aqui.** `griddata` com método linear só interpola dentro do
*fecho convexo* dos pontos amostrados; fora dele retorna `NaN`. Esses `NaN` **não são
extrapolados** — são marcados como *região não coberta pela amostragem* e desenhados com
hachura, o que é honesto: ali não houve medição. O IDW preenche todo o domínio e é
oferecido como alternativa explícita, com a ressalva de que fora do fecho convexo o valor
é inferência, não medida.

Métricas cobertas: **RSSI, SNR estimado, throughput e latência** — uma figura por métrica,
por bloco, por banda e por andar. Colormap `RdYlGn` (verde = melhor); **invertido** para
latência, onde valor alto é ruim. Mapas de RSSI recebem contornos destacados em
**−67 e −70 dBm**.

In [ ]:
# --- Métricas mapeáveis: rótulo, unidade, colormap e sentido ------------------
METRICAS_MAPA = {
    "rssi_dbm":          dict(rotulo="RSSI",            unidade="dBm",  cmap="RdYlGn",   maior_melhor=True),
    "snr_estimado_db":   dict(rotulo="SNR estimado",    unidade="dB",   cmap="RdYlGn",   maior_melhor=True),
    "throughput_tcp_mbps": dict(rotulo="Throughput TCP", unidade="Mbps", cmap="RdYlGn",  maior_melhor=True),
    "latencia_media_ms": dict(rotulo="Latência média",  unidade="ms",   cmap="RdYlGn_r", maior_melhor=False),
}

def malha(dimensoes_m, n=None):
    """Malha regular sobre as dimensões reais da planta, em metros."""
    n = n or PARAMS["resolucao_malha"]
    larg, alt = dimensoes_m
    gx = np.linspace(0.0, float(larg), int(n))
    gy = np.linspace(0.0, float(alt), int(n))
    return np.meshgrid(gx, gy)

def interpolar_griddata(x, y, z, GX, GY, metodo=None):
    """Interpolação por griddata. Fora do fecho convexo retorna NaN (não extrapola)."""
    metodo = metodo or PARAMS["metodo_interpolacao"]
    Z = griddata(np.column_stack([x, y]), z, (GX, GY), method=metodo)
    return Z, f"griddata/{metodo}"

def interpolar_idw(x, y, z, GX, GY, potencia=None, eps=1e-9):
    """Inverse Distance Weighting — cobre todo o domínio, inclusive fora do fecho convexo."""
    p = potencia or PARAMS["idw_potencia"]
    pontos = np.column_stack([np.asarray(x, float), np.asarray(y, float)])
    z = np.asarray(z, float)
    dx = GX[..., None] - pontos[:, 0]
    dy = GY[..., None] - pontos[:, 1]
    dist = np.sqrt(dx ** 2 + dy ** 2)
    exato = dist < eps
    with np.errstate(divide="ignore"):
        w = 1.0 / np.power(np.where(exato, eps, dist), p)
    Z = (w * z).sum(axis=-1) / w.sum(axis=-1)
    # onde a malha coincide com um ponto medido, devolve o valor medido
    tem_exato = exato.any(axis=-1)
    if tem_exato.any():
        idx = exato.argmax(axis=-1)
        Z = np.where(tem_exato, z[idx], Z)
    return Z, f"IDW/p={p:g}"

def mascara_nao_amostrada(Z):
    """Regiões NaN = fora do fecho convexo da amostragem."""
    return ~np.isfinite(Z)

def heatmap_metrica(bloco, cfg, df, metrica, banda, andar, metodo="griddata"):
    """Desenha um heatmap sobre a planta. Retorna (nome_do_arquivo, Z, GX, GY) ou None."""
    info = METRICAS_MAPA[metrica]
    sub = df[(df.banda_ghz == banda) & (df.andar == andar)]
    sub = sub[sub[metrica].notna() & sub.x.notna() & sub.y.notna()]
    if len(sub) < 3:
        avisar(f"bloco {bloco} | {info['rotulo']} | {banda:g} GHz | {andar}: "
               f"apenas {len(sub)} ponto(s) com coordenada e valor — heatmap exige >= 3.")
        return None

    GX, GY = malha(cfg["dimensoes_m"])
    if metodo == "IDW":
        Z, met_txt = interpolar_idw(sub.x.values, sub.y.values, sub[metrica].values, GX, GY)
    else:
        Z, met_txt = interpolar_griddata(sub.x.values, sub.y.values, sub[metrica].values, GX, GY)
    nao_amostrado = mascara_nao_amostrada(Z)
    pct_vazio = 100.0 * nao_amostrado.mean()

    larg, alt = cfg["dimensoes_m"]
    fig, ax = plt.subplots(figsize=(9.5, 6.4))

    # planta de fundo, se houver
    planta = cfg.get("planta")
    if planta and Path(planta).exists():
        img = plt.imread(str(planta))
        ax.imshow(img, extent=[0, larg, 0, alt], origin="upper", zorder=0)
        fundo_txt = Path(planta).name
    else:
        fundo_txt = "sem imagem de planta"

    # hachura nas regiões sem amostragem
    if nao_amostrado.any():
        ax.contourf(GX, GY, nao_amostrado.astype(float), levels=[0.5, 1.5],
                    colors="none", hatches=["////"], zorder=1)

    cf = ax.contourf(GX, GY, np.ma.masked_invalid(Z), levels=14,
                     cmap=info["cmap"], alpha=0.62, zorder=2)
    cb = fig.colorbar(cf, ax=ax, pad=0.02)
    cb.set_label(f"{info['rotulo']} ({info['unidade']})")

    # contornos de referência em RSSI
    if metrica == "rssi_dbm":
        niveis = [n for n in PARAMS["contornos_rssi"]
                  if np.nanmin(Z) < n < np.nanmax(Z)]
        if niveis:
            cs = ax.contour(GX, GY, np.ma.masked_invalid(Z), levels=sorted(niveis),
                            colors="black", linewidths=1.6, linestyles="--", zorder=4)
            ax.clabel(cs, fmt="%.0f dBm", fontsize=7.5)

    ax.scatter(sub.x, sub.y, s=42, c="white", edgecolors="black", linewidths=1.0, zorder=6)
    for _, r in sub.iterrows():
        ax.annotate(r.ponto_id, (r.x, r.y), fontsize=6.2, xytext=(4, 4),
                    textcoords="offset points", zorder=7)
    for nome_ap, (ax_, ay_) in (cfg.get("aps") or {}).items():
        ax.scatter([ax_], [ay_], marker="^", s=130, c="#1f4fd8",
                   edgecolors="white", linewidths=1.1, zorder=8)
        ax.annotate(nome_ap, (ax_, ay_), fontsize=7, fontweight="bold",
                    xytext=(5, -11), textcoords="offset points", zorder=8)

    ax.set_xlim(0, larg); ax.set_ylim(0, alt)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_aspect("equal")
    ax.set_title(f"{info['rotulo']} interpolado — Bloco {bloco} · {banda:g} GHz · {andar}",
                 fontsize=11, fontweight="bold")

    sentido = "verde = melhor" if info["maior_melhor"] else "verde = melhor (escala invertida: valor alto é ruim)"
    extra_rssi = ("\nLinhas tracejadas: contornos de -67 e -70 dBm (limiares de projeto)."
                  if metrica == "rssi_dbm" else "")
    extra_snr = (f"\nPiso de ruído adotado: {PARAMS['piso_ruido_dbm']:.0f} dBm (uniforme, não medido)."
                 if metrica in ("snr_estimado_db",) else "")
    texto = (
        f"O QUE MOSTRA: {info['rotulo'].lower()} interpolado sobre a planta a partir de "
        f"{len(sub)} pontos medidos (círculos brancos); triângulos azuis são os APs.\n"
        f"COMO LER: cores seguem {info['cmap']} — {sentido}. Área hachurada = fora do fecho "
        f"convexo da amostragem ({pct_vazio:.0f}% da planta): sem medição, não extrapolado."
        f"{extra_rssi}{extra_snr}\n"
        f"PARÂMETROS: interpolação {met_txt}; malha {PARAMS['resolucao_malha']}x"
        f"{PARAMS['resolucao_malha']} sobre {larg:g} x {alt:g} m; fundo: {fundo_txt}.\n"
        f"{fonte_dados(bloco, cfg, len(sub))}"
    )
    anotar_figura(ax, texto, rodape_padrao(cfg, f"interpolação: {met_txt}"), modo="externa")
    nome = salvar_fig(fig, info["rotulo"].split()[0].lower(), bloco, banda, andar)
    plt.show()
    return nome, Z, GX, GY

print("Funções de interpolação e heatmap definidas.")

In [ ]:
# --- Execução dos heatmaps para todos os blocos com dados espaciais ----------
RASTERS = {}   # (bloco, metrica, banda, andar) -> (Z, GX, GY)

def rodar_heatmaps(dados, config):
    for bloco in dados:
        df, cfg = dados[bloco], config[bloco]
        faltantes = [c for c in ("x", "y") if df[c].notna().sum() == 0]
        if faltantes:
            bloquear("5.2 Heatmaps interpolados", bloco,
                     "as leituras não têm coordenadas em planta; a posição foi registrada "
                     "apenas como descrição textual do local, e coordenadas não podem ser "
                     "inventadas a partir de texto", faltantes)
            continue
        if not cfg.get("dimensoes_m") or not cfg.get("planta"):
            bloquear("5.2 Heatmaps interpolados", bloco,
                     "faltam 'dimensoes_m' e/ou 'planta' na configuração do bloco",
                     ["dimensoes_m", "planta"])
            continue
        for metrica in METRICAS_MAPA:
            if df[metrica].notna().sum() == 0:
                bloquear(f"5.2 Heatmap de {METRICAS_MAPA[metrica]['rotulo']}", bloco,
                         "coluna vazia na campanha", [metrica])
                continue
            for banda in sorted(df.banda_ghz.dropna().unique()):
                for andar in sorted(df[df.banda_ghz == banda].andar.unique()):
                    res = heatmap_metrica(bloco, cfg, df, metrica, banda, andar)
                    if res:
                        nome, Z, GX, GY = res
                        RASTERS[(bloco, metrica, banda, andar)] = (Z, GX, GY)

rodar_heatmaps(DADOS, BLOCOS)
if not RASTERS:
    print("\nNenhum heatmap gerado — ver bloqueios acima.")

In [ ]:
# --- Interpretação da Fase 5.2 (texto para o relatório) ----------------------
print("=" * 78)
print("INTERPRETAÇÃO — Fase 5.2")
print("=" * 78)
if RASTERS:
    print(f"""
Foram gerados {len(RASTERS)} mapas interpolados. A leitura de cada mapa deve observar a
área hachurada: ela delimita a região fora do fecho convexo dos pontos medidos, onde o
griddata linear não produz valor. Essa área não foi preenchida por extrapolação — o vazio
é informação, e indica onde a amostragem precisa ser adensada antes de qualquer afirmação
sobre cobertura. Os contornos de -67 e -70 dBm nos mapas de RSSI separam a região de
operação confortável da faixa em que a taxa de modulação já começa a cair.""")
else:
    print("""
Nenhum mapa foi produzido nesta execução. O motivo é estrutural, não um defeito do código:
a campanha registrou a posição de cada ponto como descrição textual do local (por exemplo
"Fundo da M102" ou "Escada de incêndio, 1º andar"), e não como par de coordenadas medido
sobre a planta baixa. Converter texto em coordenada seria inventar dado, e a interpolação
resultante herdaria essa invenção com aparência de medição.

O que destrava a seção: uma planta baixa em imagem, as dimensões reais da edificação em
metros e uma coluna x, y por ponto, medidas no mesmo referencial. Com esses três itens
preenchidos na configuração do bloco, esta seção passa a rodar sem nenhuma alteração de
código — as funções de malha, griddata, IDW, máscara de não-amostragem e contornos de
-67/-70 dBm já estão implementadas e são exercitadas assim que houver coordenada.""")

---
# Fase 5.3 — Exportação para QGIS

Prepara os insumos para o georreferenciamento em **coordenadas locais em metros**
(origem no canto da planta):

1. `pontos_<bloco>.csv` — camada de pontos com `x`, `y` e todas as métricas, pronta para
   importar como *camada delimitada por texto*;
2. rasters interpolados — **GeoTIFF** via `rasterio` se disponível; caso contrário
   **PNG + world file `.pgw`**, que o QGIS lê nativamente;
3. `README_QGIS.md` — passo a passo do georreferenciamento.

In [ ]:
def exportar_pontos_qgis(bloco, cfg, df):
    """Camada de pontos para o QGIS. Exporta mesmo sem x/y, sinalizando a limitação."""
    cols = ["ponto_id", "bloco", "andar", "local", "x", "y", "banda_ghz", "rssi_dbm",
            "snr_estimado_db", "canal", "canal_melhor", "qualidade_melhor_canal_pct",
            "redes_vizinhas_mesmo_canal", "distancia_ao_ap", "obstaculos", "sem_obstaculo",
            "throughput_tcp_mbps", "throughput_udp_mbps", "latencia_media_ms",
            "perda_pacotes_pct"]
    saida = df[[c for c in cols if c in df.columns]].copy()
    caminho = DIR_QGIS / f"pontos_{bloco}.csv"
    saida.to_csv(caminho, index=False, encoding="utf-8")
    if saida["x"].notna().sum() == 0:
        avisar(f"bloco {bloco}: {caminho.name} exportado, mas SEM coordenadas x/y — "
               f"a camada não é plotável no QGIS até que x e y sejam preenchidos.")
    else:
        print(f"[QGIS] {caminho.name}: {len(saida)} pontos com coordenada.")
    return caminho

def exportar_raster(bloco, metrica, banda, andar, Z, cfg):
    """GeoTIFF via rasterio; senão PNG + world file .pgw. Mesma transformação afim."""
    larg, alt = cfg["dimensoes_m"]
    ny, nx = Z.shape
    px, py = larg / nx, alt / ny
    base = f"raster_{metrica}_{bloco}_{banda:g}GHz_{str(andar).replace(' ', '-')}"
    Zf = np.flipud(Z)   # linha 0 do raster = topo (y máximo)

    if TEM_RASTERIO:
        caminho = DIR_QGIS / f"{base}.tif"
        transform = from_origin(0.0, alt, px, py)   # origem no canto superior esquerdo
        with rasterio.open(caminho, "w", driver="GTiff", height=ny, width=nx, count=1,
                           dtype="float32", crs=None, transform=transform,
                           nodata=np.nan) as dst:
            dst.write(Zf.astype("float32"), 1)
        print(f"[QGIS] {caminho.name} (GeoTIFF, {nx}x{ny}, pixel {px:.3f} x {py:.3f} m)")
        return caminho

    # Fallback: PNG + world file
    caminho_png = DIR_QGIS / f"{base}.png"
    plt.imsave(caminho_png, np.ma.masked_invalid(Zf), cmap=METRICAS_MAPA[metrica]["cmap"])
    # .pgw: tamanho px x, rotação, rotação, tamanho px y (negativo), x centro px (0,0), y centro px (0,0)
    caminho_pgw = DIR_QGIS / f"{base}.pgw"
    caminho_pgw.write_text(
        f"{px:.10f}\n0.0000000000\n0.0000000000\n{-py:.10f}\n{px/2:.10f}\n{alt - py/2:.10f}\n",
        encoding="utf-8")
    print(f"[QGIS] {caminho_png.name} + {caminho_pgw.name} "
          f"(PNG + world file, {nx}x{ny}, pixel {px:.3f} x {py:.3f} m)")
    return caminho_png

# --- Execução ----------------------------------------------------------------
ARQUIVOS_QGIS = []
for _bloco in BLOCOS_ATIVOS:
    ARQUIVOS_QGIS.append(exportar_pontos_qgis(_bloco, BLOCOS[_bloco], DADOS[_bloco]))

for (_bloco, _metrica, _banda, _andar), (_Z, _GX, _GY) in RASTERS.items():
    ARQUIVOS_QGIS.append(exportar_raster(_bloco, _metrica, _banda, _andar, _Z, BLOCOS[_bloco]))

if not RASTERS:
    print("[QGIS] Nenhum raster exportado — depende dos heatmaps da Fase 5.2.")
print(f"\nMétodo de raster adotado neste ambiente: "
      f"{'GeoTIFF (rasterio disponível)' if TEM_RASTERIO else 'PNG + world file .pgw (rasterio ausente)'}")

In [ ]:
# --- README_QGIS.md ----------------------------------------------------------
metodo_raster = ("GeoTIFF (rasterio disponível neste ambiente)" if TEM_RASTERIO
                 else "PNG + world file .pgw (rasterio ausente neste ambiente)")
blocos_txt = ", ".join(BLOCOS_ATIVOS) if BLOCOS_ATIVOS else "nenhum"

readme = f"""# Georreferenciamento no QGIS — Mapa de Cobertura Wi-Fi

Gerado em {CARIMBO} pelo notebook das Fases 5 e 6.
Blocos exportados: {blocos_txt}.
Formato de raster adotado: **{metodo_raster}**.

O sistema de referência é **local, em metros**, com origem no canto da planta
(x cresce para a direita, y cresce para cima). Não há CRS geográfico associado:
trabalhe em um projeto sem projeção ou defina um CRS cartesiano genérico.

## 1. Georreferenciar a planta baixa

1. `Camada > Georreferenciador`.
2. Abra a imagem da planta.
3. Marque no mínimo 4 pontos de controle, preferencialmente os cantos do pavimento.
4. Para cada ponto, informe a coordenada local em metros. Para uma planta de
   L x A metros, os cantos são `(0, 0)`, `(L, 0)`, `(L, A)` e `(0, A)`,
   com `(0, 0)` no canto inferior esquerdo.
5. Tipo de transformação: **Linear** (ou Helmert). Reamostragem: vizinho mais próximo.
   Deixe o CRS em branco ou use um cartesiano genérico.
6. Execute. A planta passa a ocupar exatamente o retângulo `0..L` por `0..A`.

## 2. Importar a camada de pontos

1. `Camada > Adicionar camada > Adicionar camada de texto delimitado`.
2. Arquivo: `pontos_<bloco>.csv`.
3. Formato: CSV. Geometria: **coordenadas de ponto**, campo X = `x`, campo Y = `y`.
4. CRS: o mesmo usado no passo 1.
5. Estilize por `rssi_dbm` com gradação graduada, paleta RdYlGn, para reproduzir a
   leitura dos heatmaps.

> Se a coluna `x`/`y` do CSV estiver vazia, a camada carrega como tabela sem geometria.
> É o caso de qualquer bloco cuja campanha só registrou a localização como texto.

## 3. Carregar os rasters interpolados

- **GeoTIFF**: `Camada > Adicionar camada > Adicionar camada raster` e selecione o `.tif`.
  A transformação afim já está embutida; o raster cai sobre a planta.
- **PNG + `.pgw`**: mantenha o `.png` e o `.pgw` **no mesmo diretório e com o mesmo nome
  base**. Adicione o `.png` como camada raster — o QGIS lê o world file automaticamente
  e aplica a mesma transformação.

Estilize com `Propriedades > Simbologia > Banda simples falsa-cor`, paleta RdYlGn,
e reduza a opacidade para ~65% de modo que a planta permaneça visível por baixo.

## 4. Conferência

Sobreponha a camada de pontos ao raster: cada ponto medido deve cair sobre a região do
raster com o valor correspondente. Divergência indica erro de georreferenciamento no
passo 1 — refaça os pontos de controle.

## Arquivos gerados

"""
for _a in ARQUIVOS_QGIS:
    readme += f"- `{Path(_a).name}`\n"
if not ARQUIVOS_QGIS:
    readme += "- (nenhum arquivo gerado nesta execução)\n"

caminho_readme = BASE / "README_QGIS.md"
caminho_readme.write_text(readme, encoding="utf-8")
print(f"[QGIS] README gerado: {caminho_readme}")
print(f"[QGIS] {len(ARQUIVOS_QGIS)} arquivo(s) em {DIR_QGIS}")

---
# Fase 6.1 — Ajuste do modelo de path-loss

Modelo log-distância: **RSSI(d) = RSSI(d₀) − 10·α·log₁₀(d/d₀) − L_obstáculo**

Decisões metodológicas, e o motivo de cada uma:

| Decisão | Motivo |
|---|---|
| Ajuste **por bloco × banda**, nunca agregando blocos | Prédios distintos têm APs, potências e construção distintas; um α agregado não descreve nenhum dos dois |
| `andar` tratado **dentro** do bloco | "Térreo do M" e "Térreo do I" são locais diferentes e não podem cair no mesmo grupo |
| **d₀ = 1 m fixo** (não `distancia_ao_ap.min()`) | d₀ variável entre blocos torna os interceptos incomparáveis |
| Apenas pontos **sem obstáculo** alimentam o ajuste | Os obstruídos servem para medir L_obstáculo = RSSI_previsto − RSSI_medido |
| Guarda de suficiência amostral | Menos de 3 distâncias distintas ou menos de 3 pontos ⇒ α **não estimável** |
| α negativo ⇒ resultado **suprimido** | Sinal crescendo com a distância não tem significado físico; o número não é reportado |

O método simples sugerido no guia (`np.polyfit` sobre todos os pontos, d₀ = menor distância)
é calculado **em paralelo, apenas como comparação**.

In [ ]:
def ajustar_path_loss(sub, d0=None, rotulo=""):
    """Regressão de mínimos quadrados de RSSI contra 10*log10(d/d0).

    Retorna dicionário sempre — inclusive quando não é estimável, com 'ok'=False
    e o motivo. Nunca lança exceção e nunca devolve α sem R² e n.
    """
    d0 = d0 or PARAMS["d0_m"]
    sub = sub[sub["distancia_ao_ap"].notna() & sub["rssi_dbm"].notna()]
    n = len(sub)
    base = dict(rotulo=rotulo, n=n, d0=d0, ok=False, alpha=np.nan,
                rssi_d0=np.nan, r2=np.nan, motivo="", pontos=list(sub.get("ponto_id", [])))

    n_dist = sub["distancia_ao_ap"].nunique() if n else 0
    base["n_distancias"] = n_dist
    if n < PARAMS["min_pontos_ajuste"]:
        base["motivo"] = (f"amostra insuficiente: n = {n} "
                          f"(mínimo {PARAMS['min_pontos_ajuste']})")
        return base
    if n_dist < PARAMS["min_distancias_distintas"]:
        base["motivo"] = (f"amostra concentrada: apenas {n_dist} distância(s) distinta(s) "
                          f"(mínimo {PARAMS['min_distancias_distintas']}) — "
                          f"não há alavanca para separar α da dispersão")
        return base

    X = 10.0 * np.log10(sub["distancia_ao_ap"].values.astype(float) / d0)
    Y = sub["rssi_dbm"].values.astype(float)
    inclinacao, intercepto = np.polyfit(X, Y, 1)
    Yhat = intercepto + inclinacao * X
    ss_res = float(((Y - Yhat) ** 2).sum())
    ss_tot = float(((Y - Y.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    alpha = -inclinacao
    base.update(alpha=float(alpha), rssi_d0=float(intercepto), r2=float(r2), ok=True)
    if alpha < 0:
        base["ok"] = False
        base["motivo"] = ("α negativo — implicaria sinal crescendo com a distância, "
                          "resultado sem significado físico; valor suprimido")
        base["alpha_bruto"] = float(alpha)   # guardado só para diagnóstico
    return base

def ajuste_simples_do_guia(sub):
    """Método do guia: polyfit sobre TODOS os pontos, d0 = menor distância observada.
    Calculado apenas para comparação com o método rigoroso."""
    sub = sub[sub["distancia_ao_ap"].notna() & sub["rssi_dbm"].notna()]
    if len(sub) < 2 or sub["distancia_ao_ap"].nunique() < 2:
        return dict(ok=False, motivo="dados insuficientes", n=len(sub))
    d0 = float(sub["distancia_ao_ap"].min())
    r = ajustar_path_loss(sub, d0=d0, rotulo="método simples do guia (todos os pontos)")
    r["d0"] = d0
    return r

def teste_alavancagem(sub, d0=None):
    """Leave-one-out: recalcula α removendo um ponto por vez.
    Lista as observações influentes conforme os limiares de PARAMS."""
    ref = ajustar_path_loss(sub, d0=d0)
    if not ref["ok"]:
        return ref, pd.DataFrame()
    linhas = []
    for idx in sub.index:
        reduzido = sub.drop(index=idx)
        r = ajustar_path_loss(reduzido, d0=d0)
        d_alpha = r["alpha"] - ref["alpha"] if r["ok"] else np.nan
        d_r2 = r["r2"] - ref["r2"] if r["ok"] else np.nan
        influente = bool(
            (np.isfinite(d_alpha) and abs(d_alpha) > PARAMS["loo_delta_alpha"]) or
            (np.isfinite(d_r2) and -d_r2 > PARAMS["loo_queda_r2"]) or
            (not r["ok"])
        )
        linhas.append({
            "ponto_removido": sub.loc[idx, "ponto_id"],
            "d_m": float(sub.loc[idx, "distancia_ao_ap"]),
            "rssi_dbm": float(sub.loc[idx, "rssi_dbm"]),
            "alpha_sem_o_ponto": round(r["alpha"], 3) if r["ok"] else np.nan,
            "delta_alpha": round(d_alpha, 3) if np.isfinite(d_alpha) else np.nan,
            "r2_sem_o_ponto": round(r["r2"], 3) if r["ok"] else np.nan,
            "delta_r2": round(d_r2, 3) if np.isfinite(d_r2) else np.nan,
            "influente": influente,
            "observacao": "" if r["ok"] else f"ajuste inviável sem o ponto: {r['motivo']}",
        })
    return ref, pd.DataFrame(linhas).sort_values("delta_alpha", key=abs, ascending=False)

def interpretar_alpha(alpha):
    """Compara α com a faixa teórica de referência."""
    lo, hi = PARAMS["alpha_teorico"]
    if not np.isfinite(alpha):
        return "α não estimável."
    if alpha < 2.0:
        return (f"α = {alpha:.2f} está ABAIXO da faixa teórica {lo:g}–{hi:g}. Valor menor que o "
                f"espaço livre sugere guiamento de onda (corredor) ou, mais provavelmente, "
                f"amostra mal condicionada em distância.")
    if alpha <= hi:
        pos = "próximo do espaço livre" if alpha < 2.6 else (
              "típico de interior com obstrução moderada" if alpha < 3.3 else
              "típico de interior densamente obstruído")
        return (f"α = {alpha:.2f} está DENTRO da faixa teórica {lo:g}–{hi:g}, {pos}.")
    return (f"α = {alpha:.2f} está ACIMA da faixa teórica {lo:g}–{hi:g}, indicando ambiente "
            f"severamente obstruído ou mistura de percursos com sombreamento não separado.")

print("Funções de ajuste, alavancagem e interpretação definidas.")

In [ ]:
# --- Ajuste por bloco x banda ------------------------------------------------
MODELOS = {}        # (bloco, banda) -> modelo adotado
AJUSTES_DETALHE = []  # tabela longa com todos os conjuntos avaliados
ALAVANCAGEM = {}    # (bloco, banda) -> DataFrame do leave-one-out

def rodar_ajustes(dados, config):
    for bloco in dados:
        df, cfg = dados[bloco], config[bloco]
        if df["distancia_ao_ap"].notna().sum() == 0:
            bloquear("6.1 Ajuste de path-loss", bloco, "coluna de distância vazia",
                     ["distancia_ao_ap"])
            continue

        for banda in sorted(df.banda_ghz.dropna().unique()):
            faixa = df[df.banda_ghz == banda]
            livres = faixa[faixa.sem_obstaculo]
            exc = (cfg.get("exclusoes_ajuste") or {}).get(banda, {})
            excluidos = exc.get("pontos", [])
            motivo_exc = exc.get("motivo", "")
            livres_sem_exc = livres[~livres.ponto_num.isin(excluidos)]

            print("=" * 78)
            print(f"BLOCO {bloco} · {banda:g} GHz — {len(faixa)} leituras "
                  f"({len(livres)} sem obstáculo)")
            print("=" * 78)

            # conjunto 1: todos os pontos livres
            m_todos = ajustar_path_loss(livres, rotulo="todos os pontos livres")
            # conjunto 2: livres menos exclusões declaradas
            rot2 = (f"livres, excluídos {', '.join(str(p) for p in excluidos)}"
                    if excluidos else "livres (sem exclusões declaradas)")
            m_exc = ajustar_path_loss(livres_sem_exc, rotulo=rot2)
            # método do guia, para comparação
            m_guia = ajuste_simples_do_guia(faixa)

            # ajustes por andar (diagnóstico; não substituem o modelo do bloco)
            por_andar = []
            for andar in sorted(livres.andar.unique()):
                por_andar.append(ajustar_path_loss(livres[livres.andar == andar],
                                                   rotulo=f"somente {andar}"))

            for m in [m_todos, m_exc, m_guia] + por_andar:
                AJUSTES_DETALHE.append({
                    "bloco": bloco, "banda_ghz": banda, "conjunto": m.get("rotulo", "—"),
                    "n": m.get("n"), "n_distancias": m.get("n_distancias"),
                    "d0_m": m.get("d0"),
                    "alpha": round(m["alpha"], 3) if m.get("ok") else np.nan,
                    "rssi_d0_dbm": round(m["rssi_d0"], 2) if m.get("ok") else np.nan,
                    "r2": round(m["r2"], 3) if m.get("ok") else np.nan,
                    "estimavel": m.get("ok", False),
                    "motivo": m.get("motivo", ""),
                })
                est = (f"α = {m['alpha']:.2f} · RSSI(d0) = {m['rssi_d0']:.1f} dBm · "
                       f"R² = {m['r2']:.2f} · n = {m['n']}" if m.get("ok")
                       else f"NÃO ESTIMÁVEL — {m.get('motivo','')} (n = {m.get('n')})")
                print(f"  {m.get('rotulo','—'):<46} {est}")

            # modelo adotado: o conjunto com exclusões, se estimável; senão o completo
            adotado = m_exc if m_exc["ok"] else m_todos
            if adotado["ok"]:
                adotado = dict(adotado)
                adotado["excluidos"] = excluidos
                adotado["motivo_exclusao"] = motivo_exc
                MODELOS[(bloco, banda)] = adotado
                print(f"\n  >> MODELO ADOTADO: {adotado['rotulo']}")
                print(f"     {interpretar_alpha(adotado['alpha'])}")
                if excluidos:
                    print(f"     Pontos excluídos: {excluidos} — {motivo_exc}")
                ref, loo = teste_alavancagem(livres if not m_exc["ok"] else livres)
                ALAVANCAGEM[(bloco, banda)] = loo
            else:
                avisar(f"bloco {bloco} · {banda:g} GHz: expoente de perda de percurso "
                       f"NÃO ESTIMÁVEL — {adotado['motivo']}. Nenhum valor de α será "
                       f"reportado para esta combinação.", "NÃO ESTIMÁVEL")
                ALAVANCAGEM[(bloco, banda)] = pd.DataFrame()
            print()

rodar_ajustes(DADOS, BLOCOS)
TAB_AJUSTES = pd.DataFrame(AJUSTES_DETALHE)
display(TAB_AJUSTES)

In [ ]:
# --- Teste de alavancagem (leave-one-out) ------------------------------------
print("=" * 78)
print("TESTE DE ALAVANCAGEM — leave-one-out sobre o conjunto de pontos livres")
print(f"Limiares: |Δα| > {PARAMS['loo_delta_alpha']} ou queda de R² > {PARAMS['loo_queda_r2']}")
print("=" * 78)

for (_bloco, _banda), _loo in ALAVANCAGEM.items():
    print(f"\nBloco {_bloco} · {_banda:g} GHz")
    if _loo.empty:
        print("  (sem ajuste de referência — teste não aplicável)")
        continue
    display(_loo)
    infl = _loo[_loo.influente]
    if len(infl):
        print(f"  {len(infl)} observação(ões) influente(s):")
        for _, r in infl.iterrows():
            print(f"    - {r.ponto_removido} (d = {r.d_m:g} m, RSSI = {r.rssi_dbm:.0f} dBm): "
                  f"remover leva α a {r.alpha_sem_o_ponto} (Δ = {r.delta_alpha:+.2f}) "
                  f"e R² a {r.r2_sem_o_ponto} (Δ = {r.delta_r2:+.2f})")
    else:
        print("  Nenhuma observação isolada domina o ajuste.")

In [ ]:
# --- Figura: ajuste de path-loss por bloco x banda ---------------------------
def figura_path_loss(bloco, banda, cfg, df):
    faixa = df[df.banda_ghz == banda]
    livres = faixa[faixa.sem_obstaculo]
    obstr = faixa[~faixa.sem_obstaculo]
    modelo = MODELOS.get((bloco, banda))
    d0 = PARAMS["d0_m"]

    fig, ax = plt.subplots(figsize=(9.2, 6.0))
    ax.scatter(livres.distancia_ao_ap, livres.rssi_dbm, s=60, c="#2e7d32",
               edgecolors="black", linewidths=0.7, label="sem obstáculo", zorder=5)
    ax.scatter(obstr.distancia_ao_ap, obstr.rssi_dbm, s=60, c="#c62828", marker="s",
               edgecolors="black", linewidths=0.7, label="com obstáculo", zorder=5)
    for _, r in faixa.iterrows():
        ax.annotate(r.ponto_id, (r.distancia_ao_ap, r.rssi_dbm), fontsize=6.0,
                    xytext=(4, 4), textcoords="offset points")

    excluidos_ids = []
    if modelo and modelo.get("excluidos"):
        exc = faixa[faixa.ponto_num.isin(modelo["excluidos"])]
        excluidos_ids = list(exc.ponto_id)
        ax.scatter(exc.distancia_ao_ap, exc.rssi_dbm, s=190, facecolors="none",
                   edgecolors="#c62828", linewidths=1.8, linestyle="--",
                   label="excluído do ajuste", zorder=6)

    dmax = float(faixa.distancia_ao_ap.max())
    dgrid = np.linspace(d0, max(dmax * 1.15, d0 * 2), 200)

    if modelo:
        curva = modelo["rssi_d0"] - 10 * modelo["alpha"] * np.log10(dgrid / d0)
        ax.plot(dgrid, curva, "-", color="#1565c0", linewidth=2.4,
                label=f"modelo adotado: α = {modelo['alpha']:.2f}", zorder=4)
        lo, hi = PARAMS["alpha_teorico"]
        for a_ref, estilo in ((lo, ":"), (hi, ":")):
            ax.plot(dgrid, modelo["rssi_d0"] - 10 * a_ref * np.log10(dgrid / d0),
                    estilo, color="#9e9e9e", linewidth=1.2, zorder=3)
        ax.fill_between(dgrid,
                        modelo["rssi_d0"] - 10 * lo * np.log10(dgrid / d0),
                        modelo["rssi_d0"] - 10 * hi * np.log10(dgrid / d0),
                        color="#9e9e9e", alpha=0.12, zorder=2,
                        label=f"faixa teórica α = {lo:g}–{hi:g}")

    ax.set_xscale("log")
    ax.set_xlabel("distância ao AP associado (m, escala log)")
    ax.set_ylabel("RSSI (dBm)")
    ax.set_title(f"Ajuste de path-loss log-distância — Bloco {bloco} · {banda:g} GHz · todos os andares",
                 fontsize=11, fontweight="bold")
    ax.legend(loc="upper right", fontsize=7.5, framealpha=0.9)

    if modelo:
        loo = ALAVANCAGEM.get((bloco, banda), pd.DataFrame())
        n_infl = int(loo.influente.sum()) if not loo.empty else 0
        exc_txt = (f"{', '.join(excluidos_ids)} — {modelo.get('motivo_exclusao','')}"
                   if excluidos_ids else "nenhum")
        texto = (
            f"O QUE MOSTRA: RSSI medido contra distância ao AP, em escala log, com o ajuste de "
            f"mínimos quadrados sobre os pontos SEM obstáculo.\n"
            f"COMO LER: a reta azul é o modelo adotado; a faixa cinza delimita "
            f"α = {PARAMS['alpha_teorico'][0]:g}–{PARAMS['alpha_teorico'][1]:g} (referência de "
            f"interior). Quadrados vermelhos ficam ABAIXO da reta: a distância vertical até ela "
            f"é o L_obstáculo daquele ponto.\n"
            f"PARÂMETROS: α = {modelo['alpha']:.2f} · RSSI(d0) = {modelo['rssi_d0']:.1f} dBm · "
            f"R² = {modelo['r2']:.2f} · n = {modelo['n']} · d0 = {d0:g} m (fixo)\n"
            f"Pontos excluídos: {exc_txt}\n"
            f"Observações influentes (leave-one-out): {n_infl}\n"
            f"{fonte_dados(bloco, cfg, len(faixa))}"
        )
    else:
        texto = (
            f"O QUE MOSTRA: RSSI medido contra distância ao AP. NÃO há reta de ajuste porque o "
            f"expoente de perda de percurso NÃO É ESTIMÁVEL nesta combinação.\n"
            f"MOTIVO: as {int(livres.shape[0])} leituras sem obstáculo se concentram em apenas "
            f"{int(livres.distancia_ao_ap.nunique())} distância(s) distinta(s) "
            f"({', '.join(f'{v:g} m' for v in sorted(livres.distancia_ao_ap.unique()))}), abaixo "
            f"do mínimo de {PARAMS['min_distancias_distintas']} exigido. Reportar um α aqui seria "
            f"atribuir precisão inexistente.\n"
            f"COMO LER: observe a dispersão vertical dentro de uma mesma distância — ela supera a "
            f"variação esperada ENTRE distâncias, o que é a própria razão do bloqueio.\n"
            f"{fonte_dados(bloco, cfg, len(faixa))}"
        )
    anotar_figura(ax, texto, rodape_padrao(cfg, "sem interpolação (dados pontuais)"))
    salvar_fig(fig, "pathloss", bloco, banda, "todos")
    plt.show()

for _bloco in BLOCOS_ATIVOS:
    for _banda in sorted(DADOS[_bloco].banda_ghz.dropna().unique()):
        figura_path_loss(_bloco, _banda, BLOCOS[_bloco], DADOS[_bloco])

In [ ]:
# --- Figura: alavancagem (leave-one-out) -------------------------------------
def figura_alavancagem(bloco, banda, cfg, loo):
    if loo.empty:
        return
    modelo = MODELOS.get((bloco, banda))
    ref_alpha = ajustar_path_loss(
        DADOS[bloco][(DADOS[bloco].banda_ghz == banda) & DADOS[bloco].sem_obstaculo])
    if not ref_alpha["ok"]:
        return
    d = loo.sort_values("alpha_sem_o_ponto")
    cores = ["#c62828" if i else "#546e7a" for i in d.influente]

    fig, ax = plt.subplots(figsize=(9.2, 5.4))
    ax.barh(d.ponto_removido, d.alpha_sem_o_ponto, color=cores, edgecolor="black",
            linewidth=0.6, zorder=3)
    ax.axvline(ref_alpha["alpha"], color="#1565c0", linewidth=2.2, zorder=4,
               label=f"α com todos os pontos livres = {ref_alpha['alpha']:.2f}")
    ax.axvspan(ref_alpha["alpha"] - PARAMS["loo_delta_alpha"],
               ref_alpha["alpha"] + PARAMS["loo_delta_alpha"],
               color="#1565c0", alpha=0.12, zorder=2,
               label=f"tolerância ±{PARAMS['loo_delta_alpha']}")
    ax.set_xlabel("α recalculado sem o ponto")
    ax.set_ylabel("ponto removido")
    ax.set_title(f"Teste de alavancagem (leave-one-out) — Bloco {bloco} · {banda:g} GHz · todos os andares",
                 fontsize=11, fontweight="bold")
    ax.legend(loc="lower right", fontsize=7.5)

    infl = d[d.influente]
    lista = "; ".join(f"{r.ponto_removido}: α→{r.alpha_sem_o_ponto:.2f} (R²→{r.r2_sem_o_ponto:.2f})"
                      for _, r in infl.iterrows()) or "nenhuma"
    texto = (
        f"O QUE MOSTRA: o α que resultaria se cada ponto fosse removido do ajuste, um por vez.\n"
        f"COMO LER: barras vermelhas são observações INFLUENTES — sua remoção move α mais que "
        f"±{PARAMS['loo_delta_alpha']} ou derruba o R² mais que {PARAMS['loo_queda_r2']}. Quanto "
        f"mais longe da linha azul, mais o ajuste depende daquele único ponto.\n"
        f"PARÂMETROS: referência α = {ref_alpha['alpha']:.2f} · R² = {ref_alpha['r2']:.2f} · "
        f"n = {ref_alpha['n']} · d0 = {PARAMS['d0_m']:g} m\n"
        f"Influentes: {lista}\n"
        f"{fonte_dados(bloco, cfg, ref_alpha['n'])}"
    )
    anotar_figura(ax, texto, rodape_padrao(cfg, "sem interpolação (dados pontuais)"))
    salvar_fig(fig, "alavancagem", bloco, banda, "todos")
    plt.show()

for (_bloco, _banda), _loo in ALAVANCAGEM.items():
    figura_alavancagem(_bloco, _banda, BLOCOS[_bloco], _loo)

In [ ]:
# --- Atenuação por obstáculos: L_obstaculo = RSSI_previsto - RSSI_medido -----
def perdas_por_obstaculo(bloco, banda, df):
    modelo = MODELOS.get((bloco, banda))
    if modelo is None:
        return pd.DataFrame()
    obstr = df[(df.banda_ghz == banda) & (~df.sem_obstaculo) & df.distancia_ao_ap.notna()]
    if obstr.empty:
        return pd.DataFrame()
    prev = modelo["rssi_d0"] - 10 * modelo["alpha"] * np.log10(
        obstr.distancia_ao_ap.astype(float) / modelo["d0"])
    return pd.DataFrame({
        "bloco": bloco, "banda_ghz": banda,
        "ponto_id": obstr.ponto_id.values, "andar": obstr.andar.values,
        "local": obstr.local.values,
        "d_m": obstr.distancia_ao_ap.values,
        "rssi_medido_dbm": obstr.rssi_dbm.values,
        "rssi_previsto_dbm": np.round(prev.values, 1),
        "L_obstaculo_db": np.round(prev.values - obstr.rssi_dbm.values, 1),
        "obstaculos": obstr.obstaculos.values,
    }).sort_values("L_obstaculo_db", ascending=False)

TAB_OBSTACULOS = pd.concat(
    [perdas_por_obstaculo(b, bd, DADOS[b])
     for b in BLOCOS_ATIVOS for bd in sorted(DADOS[b].banda_ghz.dropna().unique())],
    ignore_index=True) if BLOCOS_ATIVOS else pd.DataFrame()

if not TAB_OBSTACULOS.empty:
    print("ATENUAÇÃO ADICIONAL POR OBSTÁCULO (aplicando o modelo adotado de cada bloco×banda)")
    display(TAB_OBSTACULOS)

    def figura_obstaculos(bloco, banda, cfg):
        sub = TAB_OBSTACULOS[(TAB_OBSTACULOS.bloco == bloco) &
                             (TAB_OBSTACULOS.banda_ghz == banda)]
        if sub.empty:
            return
        modelo = MODELOS[(bloco, banda)]
        fig, ax = plt.subplots(figsize=(9.2, 5.4))
        cores = plt.cm.YlOrRd(np.clip(sub.L_obstaculo_db / max(sub.L_obstaculo_db.max(), 1), 0.25, 0.95))
        ax.barh(sub.ponto_id, sub.L_obstaculo_db, color=cores, edgecolor="black",
                linewidth=0.7, zorder=3)
        for i, (_, r) in enumerate(sub.iterrows()):
            ax.text(r.L_obstaculo_db + 0.35, i, f"{r.L_obstaculo_db:.1f} dB  ({r.obstaculos})",
                    va="center", fontsize=7.2)
        ax.invert_yaxis()
        ax.set_xlim(0, sub.L_obstaculo_db.max() * 1.65)
        ax.set_xlabel("L_obstáculo (dB)")
        ax.set_ylabel("ponto")
        ax.set_title(f"Atenuação adicional por obstáculo — Bloco {bloco} · {banda:g} GHz · todos os andares",
                     fontsize=11, fontweight="bold")
        texto = (
            f"O QUE MOSTRA: perda além do path-loss, calculada como RSSI previsto pelo modelo "
            f"menos RSSI medido, para cada ponto com obstrução declarada.\n"
            f"COMO LER: barra maior = obstáculo mais severo. O valor depende do modelo adotado — "
            f"um α diferente desloca todas as barras.\n"
            f"PARÂMETROS: α = {modelo['alpha']:.2f} · RSSI(d0) = {modelo['rssi_d0']:.1f} dBm · "
            f"R² = {modelo['r2']:.2f} · n = {modelo['n']} · d0 = {modelo['d0']:g} m\n"
            f"Pontos excluídos do ajuste: "
            f"{', '.join(str(p) for p in modelo.get('excluidos', [])) or 'nenhum'}\n"
            f"{fonte_dados(bloco, cfg, len(sub))}"
        )
        anotar_figura(ax, texto, rodape_padrao(cfg, "sem interpolação (dados pontuais)"),
                      loc="lower right")
        salvar_fig(fig, "obstaculos", bloco, banda, "todos")
        plt.show()

    for _bloco in BLOCOS_ATIVOS:
        for _banda in sorted(DADOS[_bloco].banda_ghz.dropna().unique()):
            figura_obstaculos(_bloco, _banda, BLOCOS[_bloco])
else:
    print("Nenhuma atenuação por obstáculo calculada (depende de um modelo adotado).")

In [ ]:
# --- Interpretação da Fase 6.1 (texto para o relatório) ----------------------
print("=" * 78)
print("INTERPRETAÇÃO — Fase 6.1")
print("=" * 78)
for (_bloco, _banda), _m in MODELOS.items():
    _cfg = BLOCOS[_bloco]
    _loo = ALAVANCAGEM.get((_bloco, _banda), pd.DataFrame())
    _infl = _loo[_loo.influente] if not _loo.empty else pd.DataFrame()
    _simples = TAB_AJUSTES[(TAB_AJUSTES.bloco == _bloco) & (TAB_AJUSTES.banda_ghz == _banda) &
                           TAB_AJUSTES.conjunto.str.contains("guia")]
    print(f"""
Bloco {_bloco} · {_banda:g} GHz
{'-' * 74}
O ajuste sobre o conjunto "{_m['rotulo']}" resulta em α = {_m['alpha']:.2f},
RSSI(d0) = {_m['rssi_d0']:.1f} dBm, R² = {_m['r2']:.2f}, com n = {_m['n']} pontos e
d0 fixado em {_m['d0']:g} m. {interpretar_alpha(_m['alpha'])}""")
    if _m.get("excluidos"):
        print(f"""
Exclusões declaradas: pontos {_m['excluidos']} — {_m['motivo_exclusao']}
A exclusão está registrada em texto justamente para que o leitor possa recusá-la: sem ela,
o mesmo conjunto produz o ajuste alternativo listado na tabela de ajustes.""")
    if len(_infl):
        print(f"""
Alavancagem: {len(_infl)} observação(ões) domina(m) o ajuste. """ +
              "; ".join(f"remover {r.ponto_removido} (d = {r.d_m:g} m) leva α a "
                        f"{r.alpha_sem_o_ponto:.2f} e o R² a {r.r2_sem_o_ponto:.2f}"
                        for _, r in _infl.iterrows()) + """.
O modelo depende, portanto, de poucas leituras em distância maior. Essa é a fragilidade
central do ajuste e só se resolve com novas medições em distâncias intermediárias.""")
    else:
        print("\nAlavancagem: nenhuma observação isolada domina o ajuste.")
    if len(_simples) and pd.notna(_simples.iloc[0].alpha):
        _s = _simples.iloc[0]
        print(f"""
Comparação com o método simples do guia (polyfit sobre todos os pontos, d0 = menor
distância observada = {_s.d0_m:g} m): α = {_s.alpha:.2f}, R² = {_s.r2:.2f}, n = {_s.n}.
A diferença vem de duas escolhas: o método simples mistura pontos obstruídos com pontos
livres — atribuindo à distância uma perda que é de sombreamento — e ancora d0 em um valor
que muda de bloco para bloco, tornando os interceptos incomparáveis entre edificações.""")

_nao_estimaveis = TAB_AJUSTES[(~TAB_AJUSTES.estimavel) & TAB_AJUSTES.motivo.str.len().gt(0)]
if len(_nao_estimaveis):
    print(f"""
{'-' * 74}
Combinações sem α reportado
{'-' * 74}""")
    for _, r in _nao_estimaveis.iterrows():
        print(f"  Bloco {r.bloco} · {r.banda_ghz:g} GHz · conjunto '{r.conjunto}': {r.motivo}")
    print("""
Nesses casos nenhum número foi impresso, por decisão de método: um α calculado sobre
amostra concentrada em poucas distâncias não mede propagação, mede a dispersão local das
leituras. A correção é de campo, não de cálculo — medir em distâncias adicionais.""")
if not TAB_OBSTACULOS.empty:
    _pior = TAB_OBSTACULOS.iloc[0]
    print(f"""
{'-' * 74}
Obstáculos
{'-' * 74}
O obstáculo mais severo da campanha é o do ponto {_pior.ponto_id} ({_pior.obstaculos}),
com {_pior.L_obstaculo_db:.1f} dB de perda além do path-loss, a {_pior.d_m:g} m do AP.
Na outra ponta, {int((TAB_OBSTACULOS.L_obstaculo_db < 3).sum())} ponto(s) apresentam menos
de 3 dB de perda adicional, compatível com parede interna leve. A dispersão entre esses
extremos mostra que "obstáculo" não é categoria única: a natureza do material domina o
resultado, e agrupar tudo sob um único valor médio apagaria a diferença.""")

---
# Fase 6.2 — Análise cruzada de camadas

Classificação exigida pela Fase 6, com limiares parametrizados no topo do notebook
(`PARAMS["rssi_bom_dbm"]`, `PARAMS["throughput_baixo_mbps"]`):

| Condição | Diagnóstico |
|---|---|
| RSSI baixo **E** throughput baixo | **causa física** — distância ou sombreamento |
| RSSI bom **MAS** throughput baixo | **interferência de canal ou congestionamento**, não propagação |
| RSSI baixo **MAS** throughput aceitável | **caso a investigar** |
| RSSI bom e throughput aceitável | operação nominal |

A seção depende de `throughput_tcp_mbps`. Quando a coluna estiver vazia, a classificação é
bloqueada — e a **evidência de camada de canal** disponível (descasamento de canal e
qualidade do melhor canal) é analisada isoladamente, pois é ela que sustentaria o
diagnóstico de congestionamento.

In [ ]:
def classificar_cruzamento(df, rssi_bom=None, thr_baixo=None):
    """Classifica cada ponto no cruzamento RSSI x throughput."""
    rssi_bom = rssi_bom if rssi_bom is not None else PARAMS["rssi_bom_dbm"]
    thr_baixo = thr_baixo if thr_baixo is not None else PARAMS["throughput_baixo_mbps"]
    sub = df[df.rssi_dbm.notna() & df.throughput_tcp_mbps.notna()].copy()
    if sub.empty:
        return sub
    sinal_bom = sub.rssi_dbm >= rssi_bom
    vazao_ok = sub.throughput_tcp_mbps >= thr_baixo
    sub["classe"] = np.select(
        [~sinal_bom & ~vazao_ok, sinal_bom & ~vazao_ok, ~sinal_bom & vazao_ok],
        ["causa física (distância/sombreamento)",
         "interferência ou congestionamento",
         "investigar"],
        default="operação nominal")
    return sub

CORES_CLASSE = {
    "causa física (distância/sombreamento)": "#c62828",
    "interferência ou congestionamento":     "#ef6c00",
    "investigar":                            "#6a1b9a",
    "operação nominal":                      "#2e7d32",
}

def figura_cruzamento(bloco, banda, cfg, sub):
    rssi_bom, thr_baixo = PARAMS["rssi_bom_dbm"], PARAMS["throughput_baixo_mbps"]
    fig, ax = plt.subplots(figsize=(9.2, 6.0))
    xlo = min(sub.rssi_dbm.min() - 4, rssi_bom - 6)
    xhi = max(sub.rssi_dbm.max() + 4, rssi_bom + 6)
    ylo, yhi = 0, max(sub.throughput_tcp_mbps.max() * 1.15, thr_baixo * 1.6)
    ax.add_patch(plt.Rectangle((xlo, ylo), rssi_bom - xlo, thr_baixo - ylo,
                               color=CORES_CLASSE["causa física (distância/sombreamento)"], alpha=0.10, zorder=0))
    ax.add_patch(plt.Rectangle((rssi_bom, ylo), xhi - rssi_bom, thr_baixo - ylo,
                               color=CORES_CLASSE["interferência ou congestionamento"], alpha=0.10, zorder=0))
    ax.add_patch(plt.Rectangle((xlo, thr_baixo), rssi_bom - xlo, yhi - thr_baixo,
                               color=CORES_CLASSE["investigar"], alpha=0.10, zorder=0))
    ax.add_patch(plt.Rectangle((rssi_bom, thr_baixo), xhi - rssi_bom, yhi - thr_baixo,
                               color=CORES_CLASSE["operação nominal"], alpha=0.10, zorder=0))
    for classe, grupo in sub.groupby("classe"):
        ax.scatter(grupo.rssi_dbm, grupo.throughput_tcp_mbps, s=70,
                   c=CORES_CLASSE.get(classe, "#455a64"), edgecolors="black",
                   linewidths=0.7, label=classe, zorder=5)
    for _, r in sub.iterrows():
        ax.annotate(r.ponto_id, (r.rssi_dbm, r.throughput_tcp_mbps), fontsize=6.0,
                    xytext=(4, 4), textcoords="offset points")
    ax.axvline(rssi_bom, color="black", linestyle="--", linewidth=1.2)
    ax.axhline(thr_baixo, color="black", linestyle="--", linewidth=1.2)
    ax.set_xlim(xlo, xhi); ax.set_ylim(ylo, yhi)
    ax.set_xlabel("RSSI (dBm)"); ax.set_ylabel("throughput TCP (Mbps)")
    ax.set_title(f"Cruzamento RSSI × throughput — Bloco {bloco} · {banda:g} GHz · todos os andares",
                 fontsize=11, fontweight="bold")
    ax.legend(loc="lower right", fontsize=7.2, framealpha=0.9)
    contagem = sub.classe.value_counts().to_dict()
    texto = (
        f"O QUE MOSTRA: cada ponto medido no plano RSSI × vazão, com os quatro quadrantes de "
        f"diagnóstico coloridos ao fundo.\n"
        f"COMO LER: o quadrante em que o ponto cai indica a causa provável — sinal fraco com "
        f"vazão baixa aponta propagação; sinal bom com vazão baixa aponta disputa pelo meio, "
        f"não cobertura, e não se resolve acrescentando potência.\n"
        f"PARÂMETROS: limiar de RSSI = {rssi_bom:g} dBm · limiar de vazão = {thr_baixo:g} Mbps\n"
        f"Distribuição: " + "; ".join(f"{k}: {v}" for k, v in contagem.items()) + "\n"
        f"{fonte_dados(bloco, cfg, len(sub))}"
    )
    anotar_figura(ax, texto, rodape_padrao(cfg, "sem interpolação (dados pontuais)"))
    salvar_fig(fig, "cruzamento", bloco, banda, "todos")
    plt.show()

TAB_CRUZAMENTO = []
for _bloco in BLOCOS_ATIVOS:
    _df, _cfg = DADOS[_bloco], BLOCOS[_bloco]
    if _df.throughput_tcp_mbps.notna().sum() == 0:
        bloquear("6.2 Cruzamento RSSI x throughput", _bloco,
                 "a campanha não mediu vazão; sem ela não é possível separar limitação de "
                 "propagação de limitação por disputa do meio", ["throughput_tcp_mbps"])
        continue
    for _banda in sorted(_df.banda_ghz.dropna().unique()):
        _sub = classificar_cruzamento(_df[_df.banda_ghz == _banda])
        if _sub.empty:
            continue
        TAB_CRUZAMENTO.append(_sub[["bloco", "ponto_id", "andar", "banda_ghz", "rssi_dbm",
                                    "throughput_tcp_mbps", "classe"]])
        display(_sub[["ponto_id", "andar", "rssi_dbm", "throughput_tcp_mbps", "classe"]])
        figura_cruzamento(_bloco, _banda, _cfg, _sub)

TAB_CRUZAMENTO = (pd.concat(TAB_CRUZAMENTO, ignore_index=True)
                  if TAB_CRUZAMENTO else pd.DataFrame())

In [ ]:
# --- Evidência de camada de canal: descasamento e qualidade ------------------
# Roda mesmo sem throughput: é a evidência que sustentaria o diagnóstico de congestionamento.
def analisar_canal(bloco, df):
    linhas = []
    for banda in sorted(df.banda_ghz.dropna().unique()):
        faixa = df[df.banda_ghz == banda]
        val = faixa[faixa.canal.notna() & faixa.canal_melhor.notna()]
        n_desc = int((val.canal != val.canal_melhor).sum()) if len(val) else 0
        qual = faixa.qualidade_melhor_canal_pct
        linhas.append({
            "bloco": bloco, "banda_ghz": banda, "n_leituras": len(faixa),
            "n_comparaveis": len(val),
            "descasamento": n_desc,
            "descasamento_pct": round(100 * n_desc / len(val), 1) if len(val) else np.nan,
            "qualidade_media_pct": round(float(qual.mean()), 1) if qual.notna().any() else np.nan,
            "canal_mais_usado": (int(faixa.canal.mode().iloc[0])
                                 if faixa.canal.notna().any() else None),
            "canal_mais_indicado": (int(faixa.canal_melhor.mode().iloc[0])
                                    if faixa.canal_melhor.notna().any() else None),
        })
    return pd.DataFrame(linhas)

TAB_CANAL = (pd.concat([analisar_canal(b, DADOS[b]) for b in BLOCOS_ATIVOS], ignore_index=True)
             if BLOCOS_ATIVOS else pd.DataFrame())

if not TAB_CANAL.empty and TAB_CANAL.n_comparaveis.sum() > 0:
    print("CAMADA DE CANAL — descasamento e qualidade")
    display(TAB_CANAL)

    def figura_canal(bloco, banda, cfg, df):
        faixa = df[df.banda_ghz == banda]
        val = faixa[faixa.canal.notna() & faixa.canal_melhor.notna()]
        if val.empty:
            return
        linha = TAB_CANAL[(TAB_CANAL.bloco == bloco) & (TAB_CANAL.banda_ghz == banda)].iloc[0]
        desc = val.canal != val.canal_melhor
        fig, ax = plt.subplots(figsize=(9.4, 5.6))
        cores = ["#c62828" if d else "#2e7d32" for d in desc]
        tem_qual = val.qualidade_melhor_canal_pct.notna().any()
        alturas = (val.qualidade_melhor_canal_pct.fillna(0) if tem_qual
                   else pd.Series(np.ones(len(val)) * 50, index=val.index))
        ax.bar(val.ponto_id, alturas, color=cores, edgecolor="black", linewidth=0.6, zorder=3)
        for i, (_, r) in enumerate(val.iterrows()):
            ax.text(i, alturas.iloc[i] + 1.5, f"{int(r.canal)}→{int(r.canal_melhor)}",
                    ha="center", fontsize=6.4, rotation=90)
        ax.set_ylim(0, max(float(alturas.max()) * 1.32, 10.0))   # espaço para os rótulos
        ax.set_ylabel("qualidade do melhor canal (%)" if tem_qual else "(qualidade não medida)")
        ax.set_xlabel("ponto")
        ax.tick_params(axis="x", rotation=90, labelsize=6.5)
        ax.set_title(f"Ocupação de canal e descasamento — Bloco {bloco} · {banda:g} GHz · todos os andares",
                     fontsize=11, fontweight="bold")
        ax.legend(handles=[
            Line2D([0], [0], marker="s", color="none", markerfacecolor="#c62828",
                   markeredgecolor="black", markersize=9, label="opera em canal ≠ do melhor"),
            Line2D([0], [0], marker="s", color="none", markerfacecolor="#2e7d32",
                   markeredgecolor="black", markersize=9, label="opera no melhor canal")],
            loc="upper right", fontsize=7.5)
        texto = (
            f"O QUE MOSTRA: para cada ponto, a qualidade do melhor canal indicada pelo analisador, "
            f"colorida conforme o AP esteja ou não operando nesse canal. O rótulo sobre a barra "
            f"lê-se 'canal em uso → canal recomendado'.\n"
            f"COMO LER: barra vermelha e baixa é o pior caso — canal congestionado E diferente do "
            f"recomendado. Este é um indicador de camada de canal; ele NÃO mede propagação.\n"
            f"PARÂMETROS: descasamento = {int(linha.descasamento)} de {int(linha.n_comparaveis)} "
            f"({linha.descasamento_pct:.0f} %) · qualidade média = {linha.qualidade_media_pct} %\n"
            f"Canal mais usado: {linha.canal_mais_usado} · mais indicado: {linha.canal_mais_indicado}\n"
            f"{fonte_dados(bloco, cfg, len(val))}"
        )
        anotar_figura(ax, texto, rodape_padrao(cfg, "sem interpolação (dados pontuais)"),
                      loc="upper left")
        salvar_fig(fig, "canal", bloco, banda, "todos")
        plt.show()

    for _bloco in BLOCOS_ATIVOS:
        for _banda in sorted(DADOS[_bloco].banda_ghz.dropna().unique()):
            figura_canal(_bloco, _banda, BLOCOS[_bloco], DADOS[_bloco])

In [ ]:
# --- Interpretação da Fase 6.2 (texto para o relatório) ----------------------
print("=" * 78)
print("INTERPRETAÇÃO — Fase 6.2")
print("=" * 78)
if not TAB_CRUZAMENTO.empty:
    for _bloco in TAB_CRUZAMENTO.bloco.unique():
        _s = TAB_CRUZAMENTO[TAB_CRUZAMENTO.bloco == _bloco]
        _c = _s.classe.value_counts()
        print(f"""
Bloco {_bloco}
{'-' * 74}
Dos {len(_s)} pontos com RSSI e vazão medidos, {_c.get('interferência ou congestionamento', 0)}
apresentam sinal bom com vazão baixa — o diagnóstico aqui não é cobertura, e sim disputa
pelo meio: acrescentar potência ou APs não resolveria. {_c.get('causa física (distância/sombreamento)', 0)}
ponto(s) combinam sinal fraco e vazão baixa, caso em que a causa é de propagação e a ação
correta é posicionamento de AP. {_c.get('investigar', 0)} ponto(s) apresentam sinal fraco com
vazão ainda aceitável — merecem investigação, pois indicam que o limiar de RSSI adotado
({PARAMS['rssi_bom_dbm']:g} dBm) pode estar conservador para este ambiente.""")
else:
    print("""
A classificação cruzada não pôde ser executada: a campanha não mediu vazão. A consequência
metodológica é específica e vale registrar no relatório — sem throughput não é possível
distinguir um ponto limitado por propagação de um ponto limitado por disputa do meio, que
é justamente a distinção que esta seção existe para fazer. Os dois casos produzem a mesma
queixa do usuário e exigem intervenções opostas.""")

if not TAB_CANAL.empty and TAB_CANAL.n_comparaveis.sum() > 0:
    print(f"""
{'-' * 74}
Evidência de camada de canal (disponível independentemente da vazão)
{'-' * 74}""")
    for _, r in TAB_CANAL.iterrows():
        if not r.n_comparaveis:
            continue
        print(f"""
Bloco {r.bloco} · {r.banda_ghz:g} GHz — em {int(r.descasamento)} de {int(r.n_comparaveis)} leituras
({r.descasamento_pct:.0f} %) o AP opera em canal distinto do indicado como melhor pelo
analisador. A qualidade média do melhor canal é de {r.qualidade_media_pct} %. O canal mais
utilizado é o {r.canal_mais_usado}, enquanto o mais recomendado é o {r.canal_mais_indicado}.""")
    _b24 = TAB_CANAL[TAB_CANAL.banda_ghz == 2.4]
    _b5 = TAB_CANAL[TAB_CANAL.banda_ghz == 5.0]
    if len(_b24) and len(_b5) and pd.notna(_b24.iloc[0].qualidade_media_pct) and pd.notna(_b5.iloc[0].qualidade_media_pct):
        print(f"""
O contraste entre as faixas quantifica o congestionamento: {_b24.iloc[0].qualidade_media_pct} %
de qualidade média em 2,4 GHz contra {_b5.iloc[0].qualidade_media_pct} % em 5 GHz. Como a
intensidade de sinal bruto é adequada na maior parte dos pontos, o gargalo em 2,4 GHz é de
ocupação do meio, não de cobertura — maior ocupação aumenta a frequência de recuo no
CSMA/CA e o tempo até o meio ficar livre, reduzindo a vazão efetiva mesmo onde o RSSI é
elevado. A confirmação numérica dessa cadeia causal, porém, exige a medição de vazão que
esta campanha não realizou.""")

---
# Fase 6.3 — Verificação SINR → capacidade

Capacidade teórica de Shannon: **C = B·log₂(1 + SINR)**, com **B = 20 MHz**.

**Sobre o SNR.** O piso de ruído é um **parâmetro declarado no topo do notebook**
(`PARAMS["piso_ruido_dbm"]`), aplicado **uniformemente a todos os pontos e blocos**, e seu
valor é impresso em toda figura que dependa dele. Ele **não foi medido em campo** — é
referência térmica para canal de 20 MHz. A consequência é direta: a capacidade calculada é
um **teto teórico**, não uma previsão de vazão, porque ignora a interferência de redes
vizinhas, o overhead de MAC e a eficiência da modulação.

In [ ]:
def capacidade_shannon(snr_db, largura_hz=None):
    """C = B*log2(1+SINR), com SINR em escala linear. Retorna Mbps."""
    B = largura_hz or PARAMS["largura_canal_hz"]
    sinr_linear = 10.0 ** (np.asarray(snr_db, float) / 10.0)
    return B * np.log2(1.0 + sinr_linear) / 1e6

def figura_shannon(bloco, banda, cfg, df):
    sub = df[(df.banda_ghz == banda) & df.snr_estimado_db.notna()].copy()
    if sub.empty:
        return
    sub["capacidade_mbps"] = capacidade_shannon(sub.snr_estimado_db)
    tem_medido = sub.throughput_tcp_mbps.notna().sum() >= 2
    piso = PARAMS["piso_ruido_dbm"]
    B_mhz = PARAMS["largura_canal_hz"] / 1e6
    origem_snr = sub.snr_origem.iloc[0]

    fig, ax = plt.subplots(figsize=(9.2, 6.0))

    if tem_medido:
        m = sub[sub.throughput_tcp_mbps.notna()].copy()
        ax.scatter(m.capacidade_mbps, m.throughput_tcp_mbps, s=70, c="#1565c0",
                   edgecolors="black", linewidths=0.7, zorder=5, label="ponto medido")
        lim = max(m.capacidade_mbps.max(), m.throughput_tcp_mbps.max()) * 1.1
        ax.plot([0, lim], [0, lim], "--", color="black", linewidth=1.4, zorder=4,
                label="referência C medido = C teórico")
        # eficiência = fração da capacidade efetivamente atingida
        m["eficiencia"] = m.throughput_tcp_mbps / m.capacidade_mbps
        mediana = float(m.eficiencia.median())
        ax.plot([0, lim], [0, lim * mediana], ":", color="#ef6c00", linewidth=1.8,
                zorder=4, label=f"tendência observada ({100*mediana:.0f}% da capacidade)")
        m["desvio"] = (m.eficiencia - mediana).abs()
        piores = m.nlargest(min(3, len(m)), "desvio")
        ax.scatter(piores.capacidade_mbps, piores.throughput_tcp_mbps, s=190,
                   facecolors="none", edgecolors="#c62828", linewidths=2.0, zorder=6,
                   label="maior afastamento da tendência")
        for _, r in sub.iterrows():
            if pd.notna(r.throughput_tcp_mbps):
                ax.annotate(r.ponto_id, (r.capacidade_mbps, r.throughput_tcp_mbps),
                            fontsize=6.0, xytext=(4, 4), textcoords="offset points")
        ax.set_xlabel(f"capacidade teórica de Shannon (Mbps, B = {B_mhz:g} MHz)")
        ax.set_ylabel("throughput TCP medido (Mbps)")
        destaque = ("Destacados em vermelho: " +
                    ", ".join(f"{r.ponto_id} ({100*r.eficiencia:.0f}% da capacidade)"
                              for _, r in piores.iterrows()))
        explica = ("COMO LER: quanto mais abaixo da linha tracejada, maior a distância entre o "
                   "teto teórico e o desempenho real — essa lacuna é overhead de MAC, disputa do "
                   "meio e eficiência de modulação, não erro de medição.")
    else:
        sub = sub.sort_values("snr_estimado_db")
        ax.scatter(sub.snr_estimado_db, sub.capacidade_mbps, s=70, c="#1565c0",
                   edgecolors="black", linewidths=0.7, zorder=5, label="ponto medido")
        grid = np.linspace(sub.snr_estimado_db.min() - 2, sub.snr_estimado_db.max() + 2, 200)
        ax.plot(grid, capacidade_shannon(grid), "-", color="#1565c0", alpha=0.5,
                linewidth=1.6, zorder=3, label="C = B·log2(1 + SINR)")
        for _, r in sub.iterrows():
            ax.annotate(r.ponto_id, (r.snr_estimado_db, r.capacidade_mbps), fontsize=6.0,
                        xytext=(4, 4), textcoords="offset points")
        ax.set_xlabel(f"SNR estimado (dB) — {origem_snr}")
        ax.set_ylabel(f"capacidade teórica de Shannon (Mbps, B = {B_mhz:g} MHz)")
        destaque = ("SEM throughput medido nesta campanha: não há como confrontar teoria e "
                    "prática; o eixo vertical é teto teórico, jamais vazão esperada.")
        explica = ("COMO LER: a curva é o teto de capacidade imposto pela relação sinal-ruído. "
                   "Ela ignora interferência de vizinhos, overhead de MAC e eficiência de "
                   "modulação, de modo que a vazão real fica sistematicamente abaixo — "
                   "tipicamente entre 30 % e 50 % deste valor em redes 802.11 reais.")

    ax.set_title(f"Capacidade de Shannon — Bloco {bloco} · {banda:g} GHz · todos os andares",
                 fontsize=11, fontweight="bold")
    ax.legend(loc="upper left", fontsize=7.5, framealpha=0.9)
    texto = (
        f"O QUE MOSTRA: capacidade teórica C = B·log2(1 + SINR) calculada ponto a ponto.\n"
        f"{explica}\n"
        f"PARÂMETROS: B = {B_mhz:g} MHz · piso de ruído ADOTADO = {piso:.0f} dBm (uniforme para "
        f"todos os pontos e blocos, NÃO medido) · SNR {origem_snr}\n"
        f"{destaque}\n"
        f"{fonte_dados(bloco, cfg, len(sub))}"
    )
    anotar_figura(ax, texto, rodape_padrao(cfg, "sem interpolação (dados pontuais)"))
    salvar_fig(fig, "shannon", bloco, banda, "todos")
    plt.show()
    return sub

TAB_SHANNON = []
for _bloco in BLOCOS_ATIVOS:
    _df = DADOS[_bloco]
    if _df.throughput_tcp_mbps.notna().sum() == 0:
        bloquear("6.3 Shannon vs. throughput medido", _bloco,
                 "a campanha não mediu vazão; a capacidade teórica é calculada, mas não há "
                 "com o que confrontá-la", ["throughput_tcp_mbps"])
    for _banda in sorted(_df.banda_ghz.dropna().unique()):
        _r = figura_shannon(_bloco, _banda, BLOCOS[_bloco], _df)
        if _r is not None:
            TAB_SHANNON.append(_r[["bloco", "ponto_id", "banda_ghz", "andar", "rssi_dbm",
                                   "snr_estimado_db", "capacidade_mbps", "throughput_tcp_mbps"]])
TAB_SHANNON = pd.concat(TAB_SHANNON, ignore_index=True) if TAB_SHANNON else pd.DataFrame()
if not TAB_SHANNON.empty:
    display(TAB_SHANNON.round(1))

In [ ]:
# --- Interpretação da Fase 6.3 (texto para o relatório) ----------------------
print("=" * 78)
print("INTERPRETAÇÃO — Fase 6.3")
print("=" * 78)
print(f"""
Premissa declarada: o piso de ruído adotado é {PARAMS['piso_ruido_dbm']:.0f} dBm, aplicado
uniformemente a todos os pontos e blocos. Esse valor NÃO foi medido em campo — é a
referência de ruído térmico para um canal de {PARAMS['largura_canal_hz']/1e6:g} MHz. Toda a
capacidade calculada nesta seção herda essa premissa, e a rigor o que se computa é SNR e
não SINR: sem a varredura das redes vizinhas, o termo de interferência está ausente do
denominador. A consequência é que os valores abaixo são um TETO OTIMISTA.""")
if not TAB_SHANNON.empty:
    for _bloco in TAB_SHANNON.bloco.unique():
        for _banda in sorted(TAB_SHANNON[TAB_SHANNON.bloco == _bloco].banda_ghz.unique()):
            _s = TAB_SHANNON[(TAB_SHANNON.bloco == _bloco) & (TAB_SHANNON.banda_ghz == _banda)]
            _com = _s[_s.throughput_tcp_mbps.notna()]
            print(f"""
Bloco {_bloco} · {_banda:g} GHz
{'-' * 74}
O SNR estimado varia de {_s.snr_estimado_db.min():.0f} a {_s.snr_estimado_db.max():.0f} dB,
o que corresponde a uma capacidade teórica entre {_s.capacidade_mbps.min():.0f} e
{_s.capacidade_mbps.max():.0f} Mbps.""")
            if len(_com) >= 2:
                _ef = (_com.throughput_tcp_mbps / _com.capacidade_mbps)
                print(f"""A vazão medida corresponde, na mediana, a {100*_ef.median():.0f} % da
capacidade teórica. Os pontos que mais se afastam dessa tendência são os candidatos naturais
à discussão: um desvio para baixo indica perda que o SNR sozinho não explica — tipicamente
disputa pelo meio ou retransmissão — enquanto um desvio para cima sugere que o piso de ruído
adotado é conservador naquele local.""")
            else:
                print("""Não há vazão medida nesta combinação, de modo que a verificação da
cadeia SINR → capacidade → vazão permanece no plano teórico. O que se pode afirmar com os
dados existentes é apenas o limite superior; a diferença entre esse limite e o desempenho
real é exatamente a grandeza que a campanha não capturou.""")

---
# Fase 6.4 — Comparativo entre blocos

Executa apenas quando **dois ou mais blocos** têm dados. Com um único bloco, a seção é
pulada com aviso — sem lançar exceção.

O **aviso de comparabilidade** é emitido, e não bloqueia: campanhas em datas diferentes,
modelos de AP diferentes ou distribuições de distância muito distintas fazem com que a
diferença de α possa refletir amostragem ou infraestrutura, e não apenas a construção.

In [ ]:
def avisos_comparabilidade(config, dados, blocos):
    """Aponta o que compromete a comparação direta entre blocos. Avisa, não bloqueia."""
    msgs = []
    datas = {b: config[b].get("data_campanha") for b in blocos}
    if len(set(datas.values())) > 1:
        msgs.append(f"campanhas em datas diferentes ({datas}) — a ocupação do espectro varia "
                    f"com o momento da medição")
    modelos = {b: config[b].get("modelo_ap") for b in blocos}
    if len(set(map(str, modelos.values()))) > 1:
        msgs.append(f"modelos de AP diferentes ou não declarados ({modelos}) — potência de "
                    f"transmissão e ganho de antena distintos deslocam RSSI(d0) sem que a "
                    f"construção tenha mudado")
    if any(v is None for v in modelos.values()):
        msgs.append("ao menos um bloco não declara o modelo do AP, o que impede atribuir "
                    "diferenças de RSSI(d0) à edificação com segurança")
    faixas = {}
    for b in blocos:
        d = dados[b].distancia_ao_ap.dropna()
        if len(d):
            faixas[b] = (float(d.min()), float(d.max()), int(d.nunique()))
    if len(faixas) >= 2:
        maxs = [v[1] for v in faixas.values()]
        if max(maxs) > 2.0 * min(maxs):
            msgs.append(f"distribuições de distância muito distintas {faixas} — o bloco com "
                        f"maior alcance amostrado tende a produzir α mais bem condicionado, "
                        f"e a diferença pode ser de amostragem, não de propagação")
    return msgs

def tabela_comparativa(blocos, dados, config):
    linhas = []
    for b in blocos:
        df = dados[b]
        for banda in sorted(df.banda_ghz.dropna().unique()):
            faixa = df[df.banda_ghz == banda]
            m = MODELOS.get((b, banda))
            canal = TAB_CANAL[(TAB_CANAL.bloco == b) & (TAB_CANAL.banda_ghz == banda)]
            canal = canal.iloc[0] if len(canal) else None
            linhas.append({
                "bloco": b, "banda_ghz": banda,
                "modelo_ap": config[b].get("modelo_ap") or "não declarado",
                "data_campanha": config[b].get("data_campanha") or "—",
                "n_leituras": len(faixa),
                "alpha": round(m["alpha"], 2) if m else np.nan,
                "rssi_d0_dbm": round(m["rssi_d0"], 1) if m else np.nan,
                "r2": round(m["r2"], 2) if m else np.nan,
                "n_ajuste": m["n"] if m else np.nan,
                "rssi_min_dbm": float(faixa.rssi_dbm.min()),
                "rssi_max_dbm": float(faixa.rssi_dbm.max()),
                "throughput_medio_mbps": (round(float(faixa.throughput_tcp_mbps.mean()), 1)
                                          if faixa.throughput_tcp_mbps.notna().any() else np.nan),
                "qualidade_canal_media_pct": (canal.qualidade_media_pct if canal is not None else np.nan),
                "descasamento_canal_pct": (canal.descasamento_pct if canal is not None else np.nan),
            })
    return pd.DataFrame(linhas)

def figura_comparativa(banda, blocos, dados, config):
    presentes = [b for b in blocos if (b, banda) in MODELOS]
    if len(presentes) < 2:
        return
    fig, ax = plt.subplots(figsize=(9.2, 6.0))
    cores = plt.cm.tab10(np.linspace(0, 1, 10))
    dmax = max(float(dados[b][dados[b].banda_ghz == banda].distancia_ao_ap.max())
               for b in presentes)
    grid = np.linspace(PARAMS["d0_m"], dmax * 1.15, 200)
    resumo = []
    for i, b in enumerate(presentes):
        m = MODELOS[(b, banda)]
        livres = dados[b][(dados[b].banda_ghz == banda) & dados[b].sem_obstaculo]
        ax.scatter(livres.distancia_ao_ap, livres.rssi_dbm, s=55, color=cores[i],
                   edgecolors="black", linewidths=0.6, alpha=0.85, zorder=5)
        ax.plot(grid, m["rssi_d0"] - 10 * m["alpha"] * np.log10(grid / m["d0"]),
                "-", color=cores[i], linewidth=2.4, zorder=4,
                label=f"Bloco {b}: α = {m['alpha']:.2f}, R² = {m['r2']:.2f}, n = {m['n']}")
        resumo.append(f"{b}: α={m['alpha']:.2f} R²={m['r2']:.2f} n={m['n']}")
    ax.set_xscale("log")
    ax.set_xlabel("distância ao AP associado (m, escala log)")
    ax.set_ylabel("RSSI (dBm)")
    ax.set_title(f"Comparativo entre blocos — retas de ajuste · {banda:g} GHz · todos os andares",
                 fontsize=11, fontweight="bold")
    ax.legend(loc="upper right", fontsize=7.5)
    msgs = avisos_comparabilidade(config, dados, presentes)
    aviso_txt = ("ATENÇÃO À COMPARABILIDADE: " + "; ".join(msgs)
                 if msgs else "Sem ressalvas de comparabilidade detectadas.")
    texto = (
        f"O QUE MOSTRA: as retas de ajuste log-distância de cada bloco sobrepostas, na mesma banda.\n"
        f"COMO LER: inclinações diferentes indicam ambientes com atenuação distinta; interceptos "
        f"diferentes indicam potência ou ganho de AP distintos. Como d0 = {PARAMS['d0_m']:g} m é o "
        f"mesmo para todos os blocos, os interceptos SÃO comparáveis entre si.\n"
        f"PARÂMETROS: " + " | ".join(resumo) + "\n"
        f"{aviso_txt}\n"
        f"Fonte: blocos {', '.join(presentes)}, campanhas de "
        f"{', '.join(str(config[b].get('data_campanha')) for b in presentes)}"
    )
    anotar_figura(ax, texto, f"Gerado em {CARIMBO}  |  fontes: " +
                  ", ".join(Path(config[b]['csv']).name for b in presentes) +
                  "  |  sem interpolação (dados pontuais)")
    salvar_fig(fig, "comparativo", "-".join(presentes), banda, "todos")
    plt.show()

# --- Execução ----------------------------------------------------------------
if len(BLOCOS_ATIVOS) >= 2:
    TAB_COMPARATIVA = tabela_comparativa(BLOCOS_ATIVOS, DADOS, BLOCOS)
    print("TABELA COMPARATIVA ENTRE BLOCOS")
    display(TAB_COMPARATIVA)
    print("\nAVISOS DE COMPARABILIDADE")
    _msgs = avisos_comparabilidade(BLOCOS, DADOS, BLOCOS_ATIVOS)
    if _msgs:
        for _m in _msgs:
            avisar(_m, "COMPARABILIDADE")
        print("\nEstes são AVISOS, não bloqueios: a comparação segue, com a ressalva registrada.")
    else:
        print("  Nenhuma ressalva detectada.")
    _bandas = sorted(set().union(*[set(DADOS[b].banda_ghz.dropna().unique())
                                   for b in BLOCOS_ATIVOS]))
    for _banda in _bandas:
        figura_comparativa(_banda, BLOCOS_ATIVOS, DADOS, BLOCOS)
else:
    TAB_COMPARATIVA = pd.DataFrame()
    avisar(f"apenas {len(BLOCOS_ATIVOS)} bloco com dados ({', '.join(BLOCOS_ATIVOS) or 'nenhum'}) — "
           f"seção comparativa PULADA. Ela roda automaticamente assim que um segundo bloco "
           f"for preenchido na configuração.", "SEÇÃO PULADA")

In [ ]:
# --- Interpretação da Fase 6.4 (texto para o relatório) ----------------------
print("=" * 78)
print("INTERPRETAÇÃO — Fase 6.4")
print("=" * 78)
if len(BLOCOS_ATIVOS) >= 2:
    for _banda in sorted(TAB_COMPARATIVA.banda_ghz.unique()):
        _s = TAB_COMPARATIVA[(TAB_COMPARATIVA.banda_ghz == _banda) & TAB_COMPARATIVA.alpha.notna()]
        if len(_s) < 2:
            print(f"\n{_banda:g} GHz: menos de dois blocos com α estimável — comparação de "
                  f"expoente não realizada nesta banda.")
            continue
        _amp = _s.alpha.max() - _s.alpha.min()
        print(f"""
{_banda:g} GHz
{'-' * 74}
Os expoentes variam de {_s.alpha.min():.2f} a {_s.alpha.max():.2f} (amplitude de {_amp:.2f}) entre
os blocos comparados. A leitura correta dessa diferença depende das ressalvas de
comparabilidade impressas acima: antes de atribuí-la à construção, é preciso descartar que
ela venha de datas de campanha distintas, de modelos de AP diferentes ou de faixas de
distância amostradas de forma desigual.""")
else:
    print(f"""
A seção comparativa não foi executada porque há apenas {len(BLOCOS_ATIVOS)} bloco com dados
nesta execução. Isso é comportamento previsto, não falha: o notebook descobre os blocos a
partir das chaves do dicionário de configuração e ajusta o escopo da análise ao que existe.

Assim que o segundo bloco for preenchido, esta seção passa a produzir automaticamente a
figura com as retas sobrepostas por banda, a tabela comparativa e os avisos de
comparabilidade — sem nenhuma alteração no código de análise.""")

---
# Síntese e verificação

Fecha a execução com: o teste de regressão contra os valores já validados, o inventário de
figuras, a lista de análises bloqueadas e as colunas que destravam cada uma.

In [ ]:
# =============================================================================
# TESTE DE REGRESSÃO — obtido vs. esperado
# =============================================================================
def _obter(f, default=np.nan):
    try:
        v = f()
        return v if v is not None else default
    except Exception:
        return default

CASOS = []
_B = "M"   # bloco de referência do teste; ausente => teste é pulado, não falha

if _B in DADOS:
    _df = DADOS[_B]
    _livres24 = _df[(_df.banda_ghz == 2.4) & _df.sem_obstaculo]
    _exc = BLOCOS[_B]["exclusoes_ajuste"].get(2.4, {}).get("pontos", [])
    _m_exc = ajustar_path_loss(_livres24[~_livres24.ponto_num.isin(_exc)])
    _m_todos = ajustar_path_loss(_livres24)
    _m5 = MODELOS.get((_B, 5.0))
    _loo24 = ALAVANCAGEM.get((_B, 2.4), pd.DataFrame())
    _canal24 = TAB_CANAL[(TAB_CANAL.bloco == _B) & (TAB_CANAL.banda_ghz == 2.4)]
    _obst5 = TAB_OBSTACULOS[(TAB_OBSTACULOS.bloco == _B) &
                            (TAB_OBSTACULOS.ponto_id == f"{_B}-05")]

    CASOS += [
        ("α em 2,4 GHz, excluindo P13/P14", "2.62",
         _obter(lambda: f"{_m_exc['alpha']:.2f}")),
        ("RSSI(d0) em 2,4 GHz, excluindo P13/P14", "-28.5",
         _obter(lambda: f"{_m_exc['rssi_d0']:.1f}")),
        ("R² em 2,4 GHz, excluindo P13/P14", "0.91",
         _obter(lambda: f"{_m_exc['r2']:.2f}")),
        ("n em 2,4 GHz, excluindo P13/P14", "9",
         _obter(lambda: f"{_m_exc['n']}")),
        ("α em 2,4 GHz, todos os pontos livres", "2.11",
         _obter(lambda: f"{_m_todos['alpha']:.2f}")),
        ("R² em 2,4 GHz, todos os pontos livres", "0.64",
         _obter(lambda: f"{_m_todos['r2']:.2f}")),
        ("α em 5 GHz", "não estimável",
         "não estimável" if _m5 is None else f"{_m5['alpha']:.2f}"),
        ("L_obstáculo do ponto 5 (dB)", "21.1",
         _obter(lambda: f"{_obst5.iloc[0].L_obstaculo_db:.1f}")),
        ("Descasamento de canal em 2,4 GHz", "13 de 17 (76%)",
         _obter(lambda: f"{int(_canal24.iloc[0].descasamento)} de "
                        f"{int(_canal24.iloc[0].n_comparaveis)} "
                        f"({_canal24.iloc[0].descasamento_pct:.0f}%)")),
        ("Alavancagem: α sem o ponto 15", "1.61",
         _obter(lambda: f"{_loo24[_loo24.ponto_removido == f'{_B}-15'].iloc[0].alpha_sem_o_ponto:.2f}")),
        ("Alavancagem: R² sem o ponto 15", "0.30",
         _obter(lambda: f"{_loo24[_loo24.ponto_removido == f'{_B}-15'].iloc[0].r2_sem_o_ponto:.2f}")),
    ]

if CASOS:
    TAB_REGRESSAO = pd.DataFrame(CASOS, columns=["item", "esperado", "obtido"])
    TAB_REGRESSAO["situacao"] = np.where(
        TAB_REGRESSAO.esperado.str.strip() == TAB_REGRESSAO.obtido.astype(str).str.strip(),
        "OK", "DIVERGENTE")
    print("=" * 78)
    print("TESTE DE REGRESSÃO")
    print("=" * 78)
    display(TAB_REGRESSAO)
    _falhas = TAB_REGRESSAO[TAB_REGRESSAO.situacao != "OK"]
    if len(_falhas):
        avisar(f"{len(_falhas)} item(ns) do teste de regressão DIVERGEM do esperado. "
               f"Investigue antes de usar os resultados.", "REGRESSÃO")
        display(_falhas)
    else:
        print(f"\nTodos os {len(TAB_REGRESSAO)} itens reproduzem os valores validados.")
else:
    TAB_REGRESSAO = pd.DataFrame()
    avisar("bloco de referência do teste de regressão ausente — teste pulado "
           "(não é falha: o notebook roda com os blocos que existirem).", "REGRESSÃO")

In [ ]:
# =============================================================================
# INVENTÁRIO DE SAÍDAS
# =============================================================================
print("=" * 78)
print(f"FIGURAS GERADAS ({len(FIGURAS_GERADAS)}) — PNG 150 dpi em {DIR_FIG}")
print("=" * 78)
for _i, _n in enumerate(sorted(FIGURAS_GERADAS), 1):
    print(f"  {_i:2d}. {_n}")
if not FIGURAS_GERADAS:
    print("  (nenhuma)")

print("\n" + "=" * 78)
print(f"ARQUIVOS PARA QGIS ({len(ARQUIVOS_QGIS)}) em {DIR_QGIS}")
print("=" * 78)
for _a in ARQUIVOS_QGIS:
    print(f"  - {Path(_a).name}")
print(f"  - README_QGIS.md (em {BASE})")

print("\n" + "=" * 78)
print(f"ANÁLISES BLOQUEADAS ({len(ANALISES_BLOQUEADAS)})")
print("=" * 78)
if ANALISES_BLOQUEADAS:
    TAB_BLOQUEIOS = pd.DataFrame(ANALISES_BLOQUEADAS).drop_duplicates()
    display(TAB_BLOQUEIOS)
else:
    TAB_BLOQUEIOS = pd.DataFrame()
    print("  (nenhuma)")

print("\n" + "=" * 78)
print("O QUE PREENCHER PARA DESTRAVAR CADA ANÁLISE")
print("=" * 78)
if not TAB_HABILITACAO.empty:
    _blq = TAB_HABILITACAO[TAB_HABILITACAO.situacao == "BLOQUEADA"]
    if len(_blq):
        for _bloco in _blq.bloco.unique():
            print(f"\nBloco {_bloco}:")
            for _, r in _blq[_blq.bloco == _bloco].iterrows():
                print(f"  {r.analise:<34} <- preencher: {r.colunas_faltantes}")
        _todas = sorted(set(", ".join(_blq.colunas_faltantes).replace(", ", ",").split(",")))
        print(f"\nConjunto mínimo de colunas a coletar: {', '.join(_todas)}")
    else:
        print("  Todas as análises estão habilitadas.")

print("\n" + "=" * 78)
print(f"AVISOS EMITIDOS ({len(AVISOS)})")
print("=" * 78)
for _a in AVISOS:
    print(f"  {_a}")
if not AVISOS:
    print("  (nenhum)")

In [ ]:
# =============================================================================
# SÍNTESE FINAL — texto para o relatório
# =============================================================================
print("=" * 78)
print("SÍNTESE")
print("=" * 78)
print(f"""
Esta execução processou {len(BLOCOS_ATIVOS)} de {len(BLOCOS)} bloco(s) declarado(s) na
configuração ({', '.join(BLOCOS_ATIVOS) if BLOCOS_ATIVOS else 'nenhum'}), produzindo
{len(FIGURAS_GERADAS)} figura(s) e {len(ARQUIVOS_QGIS)} arquivo(s) de insumo para o QGIS.
""")

for _bloco in BLOCOS_ATIVOS:
    _df, _cfg = DADOS[_bloco], BLOCOS[_bloco]
    _ms = {bd: m for (b, bd), m in MODELOS.items() if b == _bloco}
    _alpha_txt = ("; ".join(f"{bd:g} GHz: α = {m['alpha']:.2f} (R² = {m['r2']:.2f}, n = {m['n']})"
                            for bd, m in sorted(_ms.items()))
                  if _ms else "nenhum expoente estimável com a amostra atual")
    _bandas_sem = [f"{bd:g} GHz" for bd in sorted(_df.banda_ghz.dropna().unique())
                   if (_bloco, bd) not in MODELOS]
    print(f"""Bloco {_bloco}
{'-' * 74}
{len(_df)} leituras válidas, campanha de {_cfg.get('data_campanha', 'data não informada')},
RSSI de {_df.rssi_dbm.min():.0f} a {_df.rssi_dbm.max():.0f} dBm.
Propagação — {_alpha_txt}.""")
    if _bandas_sem:
        print(f"Sem expoente reportado em: {', '.join(_bandas_sem)} — a amostra não sustenta a "
              f"estimativa,\ne nenhum número foi impresso no lugar dela.")
    _c = TAB_CANAL[TAB_CANAL.bloco == _bloco]
    for _, r in _c.iterrows():
        if r.n_comparaveis:
            print(f"Canal — {r.banda_ghz:g} GHz: {int(r.descasamento)}/{int(r.n_comparaveis)} "
                  f"({r.descasamento_pct:.0f}%) fora do melhor canal, "
                  f"qualidade média {r.qualidade_media_pct}%.")
    print()

print(f"""{'-' * 74}
Estrutura multi-bloco
{'-' * 74}
Os blocos processados vieram das chaves do dicionário de configuração, não de lista fixa.
Nenhuma função de cálculo contém nome de bloco escrito diretamente: bloco é sempre
parâmetro. Um bloco declarado sem CSV disponível é ignorado com aviso e as seções
comparativas são puladas — o notebook conclui a execução em qualquer caso.

Para incluir o Bloco I, preencha a entrada indicada na seção ">>> ADICIONAR BLOCO I AQUI <<<"
e execute o notebook novamente. Todas as análises habilitadas pelas colunas presentes
passam a rodar para os dois blocos, e a Fase 6.4 se ativa sozinha.

{'-' * 74}
Limitações desta execução
{'-' * 74}""")
if not TAB_BLOQUEIOS.empty:
    for _analise in TAB_BLOQUEIOS.analise.unique():
        _r = TAB_BLOQUEIOS[TAB_BLOQUEIOS.analise == _analise].iloc[0]
        print(f"- {_analise}: {_r.motivo}.")
else:
    print("- Nenhuma análise ficou bloqueada.")
print(f"""
Nenhum valor foi arbitrado no lugar de dado ausente, e nenhum α foi reportado sem R², n e a
verificação de suficiência amostral. Onde a amostra não sustentava a estimativa, o resultado
declarado é a impossibilidade — não um número.

Execução concluída em {CARIMBO}.""")